# Cost-aware gating under extreme class imbalance — experiment notebook (v4, major revision MLWA-D-26-02065)

v4 keeps every section of v3.1 and adds the analyses requested by the reviewers. Run it on a **Colab GPU runtime** with `Runtime -> Run all`. Long sections save checkpoints to `RESULTS_DIR/cache/`, so after a disconnect, `Run all` again resumes where it stopped. Frozen features are re-used from the v3.1 run when the frames are unchanged.

| Reviewer request | Section |
|---|---|
| Speed-up aggregated as S = 1/(rho + mean q) (R1.9, R1.10) | 8 |
| Per-fold PR-AUC, recording-cluster bootstrap 95% CIs, Clopper-Pearson recall bounds, corrected resampled t-tests (R1.6, R2.10, R3) | 15 |
| Nested evaluation of the complete model-selection procedure (R3, Cawley & Talbot 2010) | 16 |
| Threshold stability, split-conformal threshold, alternative rules (R1.1, R2.5, R3) | 17 |
| Unseen stations: few-shot calibration with separate calibration/test recordings, augmentation (R2.3, R3) | 18, 19 |
| Robustness to illumination, glare, noise, blur, viewpoint, compression (R2.9) | 19 |
| Baselines: random sampling, HOG and image-statistics gates, fine-tuned CNN gate (R2.4) | 4, 6, 20 |
| HED on the GPU (true end-to-end GPU pipeline), GPU-best validated end to end, all repetitions, stream replay (R1.5, R1.9, R2.6, R3) | 21, 22 |
| Edge-map fidelity of gating vs cheaper edge detectors (R2.8, R3) | 23 |
| Speed-up versus prevalence (R3) | 24 |
| Release files: fold assignments, per-fold results, hyperparameters (R1.3, R3) | 25 |

All outputs go to `RESULTS_DIR` (`tables/`, `figures/`, `logs/`, `cache/`, `overleaf/` + `overleaf_v4.zip`, `release/`, `results_summary_v4.json`). Expected run time on a T4 runtime: about 1.5-2.5 h.

## 1. Environment

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
# Kaggle sets KAGGLE_KERNEL_RUN_TYPE; Colab images can also contain a /kaggle folder, so Colab always wins
IN_KAGGLE = (not IN_COLAB) and bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("platform:", "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "other"))
SMOKE_TEST = os.environ.get("MLWA_SMOKE_TEST", "0") == "1"   # quick dry run only; never for the paper
if IN_COLAB and not SMOKE_TEST:
    from google.colab import drive
    drive.mount("/content/drive")

REQUIRED_OPENCV = "4.12.0.88"
def _opencv_has_caffe():
    try:
        import cv2
        return hasattr(cv2.dnn, "readNetFromCaffe") and hasattr(cv2, "dnn_registerLayer")
    except ImportError:
        return False
if not _opencv_has_caffe():
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python", "opencv-contrib-python",
                    "opencv-python-headless", "opencv-contrib-python-headless"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"opencv-python-headless=={REQUIRED_OPENCV}"], check=True)
    print(">>> OpenCV 4.x installed. The runtime restarts now; then choose 'Run all' again. <<<")
    if IN_KAGGLE:
        raise SystemExit("Kaggle: choose Run > Restart & clear cell outputs, then Run all.")
    if IN_COLAB:
        os.kill(os.getpid(), 9)
    raise SystemExit("Restart the kernel and run again.")
print("OpenCV with Caffe importer available.")

In [ ]:
import json, time, re, platform, shutil, datetime, warnings, itertools
import numpy as np, pandas as pd, cv2, matplotlib, matplotlib.pyplot as plt
import tensorflow as tf, keras
from scipy.optimize import minimize
from scipy.stats import beta
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from IPython.display import display
warnings.filterwarnings("ignore", category=UserWarning)

def _sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        return ""
def _cpu_model():
    try:
        return next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
    except Exception:
        return platform.processor() or "unknown"

GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
ENV = {"timestamp_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
       "python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
       "opencv": cv2.__version__, "numpy": np.__version__, "cpu_model": _cpu_model(), "cpu_logical_cores": os.cpu_count(),
       "gpus": [g.name for g in GPUS], "gpu_model": _sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"),
       "opencv_cuda_devices": int(cv2.cuda.getCudaEnabledDeviceCount()) if hasattr(cv2, "cuda") else 0,
       "opencv_threads": cv2.getNumThreads(), "in_colab": IN_COLAB, "in_kaggle": IN_KAGGLE,
       "platform": "Kaggle" if IN_KAGGLE else ("Google Colaboratory" if IN_COLAB else platform.node()), "smoke_test": SMOKE_TEST}
for k, v in ENV.items():
    print(f"{k:>20}: {v}")

## 2. Configuration
`IMG_SIZE` is (height, width), as in the earlier notebooks. `SCALES` adds reduced input resolutions as a cheap way to lower gate cost.

In [ ]:
BASE_DIR = os.environ.get("MLWA_BASE_DIR", "/content/drive/MyDrive/M2Project")
CONFIG = {
    "DATA_DIRS": [os.path.join(BASE_DIR, d) for d in ("train_resized", "valid_resized", "test_resized")],
    "IMG_SIZE": (300, 146),
    "BACKBONES": ["MobileNetV3Small", "MobileNetV3Large", "EfficientNetB0", "ResNet50"],
    "SCALES": [1.0, 0.5],
    "WEIGHTS": "imagenet",
    "STRATEGIES": ["LR", "LR-weighted", "LR-SMOTE", "Focal-weighted", "OneClass-Mahalanobis", "OneClass-kNN"],
    "FLIP_AUG": True,              # add mirrored training frames to supervised heads
    "C": 1.0,                      # inverse L2 strength of logistic heads
    "FOCAL_GAMMA": 2.0, "FOCAL_ALPHA": 0.25, "FOCAL_L2": 1e-3,
    "PCA_DIM": 32, "KNN_K": 5, "SMOTE_K": 5,
    "RECALL_TARGETS": [1.0, 0.95],
    "N_FOLDS": 5, "INNER_FOLDS": 3, "CV_SEEDS": [0, 1, 2],
    "LEGACY_TAU": 0.35,
    "LAT_WARMUP": 5, "LAT_REPEATS": 50, "HED_REPEATS": 20, "LAT_ROUNDS": 5,
    "HED_PROTOTXT": os.environ.get("MLWA_HED_PROTOTXT", "/content/drive/MyDrive/Holistically-Nested_Edge_Detection/deploy.prototxt"),
    "HED_MODEL": os.environ.get("MLWA_HED_MODEL", "/content/drive/MyDrive/Holistically-Nested_Edge_Detection/hed_pretrained_bsds.caffemodel"),
    "HED_MEAN": (104.00698793, 116.66876762, 122.67891434),
    "LEGACY_V2_TABLES": os.path.join(BASE_DIR, "results_mlwa", "tables"),
    "RESULTS_DIR": os.path.join(BASE_DIR, "results_mlwa_v4"),
    "FEATURE_CACHE": os.path.join(BASE_DIR, "results_mlwa_v3", "cache"),   # re-use v3.1 features when frames are unchanged
    "SEED": 1337,
    # ---- v4 (revision) ----
    "HANDCRAFTED": ["HOG", "ImgStats"],     # non-CNN baseline gates (CPU only)
    "NESTED_SEEDS": [0, 1, 2], "NESTED_INNER_FOLDS": 4,
    "BOOTSTRAP": 2000,
    "PERTURBATIONS": ["brightness-30", "brightness+30", "gamma0.6", "gamma1.6", "contrast0.6", "noise10", "noise20",
                      "glare", "motion_blur", "defocus", "rotate5", "scale_shift", "jpeg30"],
    "AUG_COPIES": 2,                        # random photometric/geometric copies per training frame (Section 19)
    "CALIB_SPLITS": 20,                     # random calibration/test splits of a held-out station's recordings
    "FINETUNE": False, "FT_SEEDS": [0], "FT_EPOCHS": 30, "FT_LR": 1e-4, "FT_BATCH": 16,
    "EDGE_TOL_PX": 2,
    "CAMERA_FPS": [5, 10, 25],
    "PREVALENCES": [0.01, 0.02, 0.05, 0.10, 0.20],
    "RESUME": True,
}
if SMOKE_TEST:
    CONFIG.update({"WEIGHTS": None, "CV_SEEDS": [0], "LAT_REPEATS": 3, "HED_REPEATS": 2, "LAT_WARMUP": 1,
                   "BACKBONES": ["MobileNetV3Small", "ResNet50"], "NESTED_SEEDS": [0], "BOOTSTRAP": 50,
                   "PERTURBATIONS": ["noise10", "glare"], "CALIB_SPLITS": 2, "FT_EPOCHS": 1, "RESUME": False, "LAT_ROUNDS": 2})

if IN_KAGGLE and not SMOKE_TEST:
    # Kaggle: inputs are read-only under /kaggle/input (attach the data, the HED files and, optionally, the v3 feature cache and
    # a previous v4 output as datasets); everything is written to /kaggle/working.
    def _find(name, want_dir=True):
        for root, dirs, files in os.walk("/kaggle/input"):
            if (want_dir and name in dirs) or (not want_dir and name in files):
                return os.path.join(root, name)
        return None
    def _find_ext(ext):
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(ext):
                    return os.path.join(root, f)
        return None
    CONFIG["DATA_DIRS"] = [_find(d) or d for d in ("train_resized", "valid_resized", "test_resized")]
    CONFIG["HED_PROTOTXT"] = _find("deploy.prototxt", want_dir=False) or CONFIG["HED_PROTOTXT"]
    CONFIG["HED_MODEL"] = _find_ext(".caffemodel") or CONFIG["HED_MODEL"]
    v3c = [os.path.join(r, "cache") for r, d, f in os.walk("/kaggle/input") if r.endswith("results_mlwa_v3") and "cache" in d]
    CONFIG["FEATURE_CACHE"] = v3c[0] if v3c else "/kaggle/working/none"
    lg = [os.path.join(r, "tables") for r, d, f in os.walk("/kaggle/input") if r.endswith("results_mlwa") and "tables" in d]
    CONFIG["LEGACY_V2_TABLES"] = lg[0] if lg else "/kaggle/working/none"
    CONFIG["RESULTS_DIR"] = "/kaggle/working/results_mlwa_v4"
    print("Kaggle paths:", {k: CONFIG[k] for k in ["DATA_DIRS", "HED_PROTOTXT", "HED_MODEL", "FEATURE_CACHE", "LEGACY_V2_TABLES", "RESULTS_DIR"]})
    missing = [d for d in CONFIG["DATA_DIRS"] + [CONFIG["HED_PROTOTXT"], CONFIG["HED_MODEL"]] if not os.path.exists(d)]
    if missing:
        raise FileNotFoundError(f"Attach the data/HED dataset (Add Input). Not found: {missing}")
RD = CONFIG["RESULTS_DIR"]
DIRS = {k: os.path.join(RD, k) for k in ["tables", "figures", "logs", "cache", "overleaf", "release"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
if IN_KAGGLE and not SMOKE_TEST:
    # resume across Kaggle sessions: copy checkpoints of an earlier run that was attached as input (its results_mlwa_v4/cache)
    prev = [os.path.join(r, "cache") for r, d, f in os.walk("/kaggle/input") if r.endswith("results_mlwa_v4") and "cache" in d]
    for pc in prev[:1]:
        for f in os.listdir(pc):
            if f.startswith("ckpt_") and not os.path.exists(os.path.join(DIRS["cache"], f)):
                shutil.copy(os.path.join(pc, f), os.path.join(DIRS["cache"], f))
        print("checkpoints copied from", pc)
json.dump(CONFIG, open(os.path.join(DIRS["logs"], "config.json"), "w"), indent=2)
json.dump(ENV, open(os.path.join(DIRS["logs"], "environment.json"), "w"), indent=2)
keras.utils.set_random_seed(CONFIG["SEED"])

RESULTS = {}
def record(k, v):
    RESULTS[k] = v
def save_table(df, name, show=True, index=False):
    df.to_csv(os.path.join(DIRS["tables"], f"{name}.csv"), index=index)
    if show:
        display(df)
    print(f"[saved] tables/{name}.csv")
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(DIRS["figures"], f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    print(f"[saved] figures/{name}.png/.pdf")
def clopper_pearson(k, n, a=0.05):
    if n == 0:
        return (np.nan, np.nan)
    return (0.0 if k == 0 else float(beta.ppf(a / 2, k, n - k + 1)), 1.0 if k == n else float(beta.ppf(1 - a / 2, k + 1, n - k)))
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

def cached(name, fn):
    # checkpoint for long sections: re-running the notebook resumes from finished sections
    p = os.path.join(DIRS["cache"], f"ckpt_{name}.pkl")
    if CONFIG["RESUME"] and os.path.exists(p):
        print(f"[resume] {name} loaded from checkpoint"); return pd.read_pickle(p)
    out = fn(); pd.to_pickle(out, p); return out

## 3. Data: pool all labelled frames, derive recording and station groups
File names look like `img_<station>_<recording>_<frame>.jpg`. A *recording* is `<station>_<recording>`; it is the grouping unit for cross-validation. The original train/val/test split is kept for reference only.

In [ ]:
def load_dir(d):
    ds = tf.keras.utils.image_dataset_from_directory(d, label_mode="binary", image_size=CONFIG["IMG_SIZE"],
                                                     batch_size=32, shuffle=False)
    X = np.concatenate([x.numpy() for x, _ in ds]).astype(np.float32)
    y = np.concatenate([t.numpy() for _, t in ds]).astype(int).ravel()
    return X, y, list(ds.file_paths), list(ds.class_names)

parts = [load_dir(d) for d in CONFIG["DATA_DIRS"]]
X = np.concatenate([p[0] for p in parts]); y = np.concatenate([p[1] for p in parts])
paths = sum([p[2] for p in parts], [])
orig_split = np.concatenate([[s] * len(p[1]) for s, p in zip(["train", "valid", "test"], parts)])
print("class folders:", parts[0][3], "-> label 1 = joint")

NAME_RE = re.compile(r"img_([A-Za-z0-9]+)_([A-Za-z0-9]+)_(\d+)")   # v3.1: also matches ids such as SIS001522
def parse(p):
    m = NAME_RE.search(os.path.basename(p))
    return (f"{m.group(1)}_{m.group(2)}", m.group(1), int(m.group(3))) if m else (os.path.basename(p), "unknown", -1)
meta = pd.DataFrame([parse(p) for p in paths], columns=["recording", "station", "frame_no"])
meta["file"] = [os.path.basename(p) for p in paths]; meta["label"] = y; meta["orig_split"] = orig_split
groups, stations = meta.recording.values, meta.station.values
save_table(meta, "frames", show=False)

summary = pd.DataFrame({
    "frames": [len(y)], "joints": [int(y.sum())], "background": [int((y == 0).sum())],
    "joint_rate_%": [round(100 * y.mean(), 1)], "recordings": [meta.recording.nunique()],
    "recordings_with_joints": [meta[meta.label == 1].recording.nunique()], "stations": [meta.station.nunique()]})
save_table(summary, "dataset_summary")
st = meta.groupby("station").label.agg(frames="size", joints="sum").reset_index()
save_table(st, "frames_per_station")
PI = float(y.mean())
record("dataset", summary.iloc[0].to_dict()); record("pi", PI)

## 4. Frozen ImageNet features (cached)
For every backbone and input scale: global-average-pooled features of each frame and of its horizontal mirror.

In [ ]:
BACKBONES = {
    "MobileNetV3Small": (tf.keras.applications.MobileNetV3Small, lambda x: x),           # rescaling is built in
    "MobileNetV3Large": (tf.keras.applications.MobileNetV3Large, lambda x: x),
    "EfficientNetB0":   (tf.keras.applications.EfficientNetB0, lambda x: x),
    "ResNet50":         (tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input),
}
def input_shape(scale):
    h, w = CONFIG["IMG_SIZE"]
    return (max(32, int(round(h * scale))), max(32, int(round(w * scale))), 3)

def build_extractor(name, scale):
    ctor, prep = BACKBONES[name]
    shp = input_shape(scale)
    inp = keras.Input(shape=shp)
    kw = dict(input_shape=shp, include_top=False, weights=CONFIG["WEIGHTS"], pooling="avg")
    if name.startswith("MobileNetV3"):
        kw["include_preprocessing"] = True
    base = ctor(**kw)
    return keras.Model(inp, base(prep(inp)), name=f"{name}_{scale}")

def resize_batch(Xb, scale):
    return Xb if scale == 1.0 else tf.image.resize(Xb, input_shape(scale)[:2], method="bilinear").numpy()

def make_fwd(model):
    @tf.autograph.experimental.do_not_convert
    def _f(x):
        return model(x, training=False)
    return tf.function(_f, reduce_retracing=True)

CONFIGS = [(b, s) for b in CONFIG["BACKBONES"] for s in CONFIG["SCALES"]]
FEATS = {}
FEAT_DIR = CONFIG["FEATURE_CACHE"] if os.path.isdir(CONFIG["FEATURE_CACHE"]) else DIRS["cache"]
FEAT_OUT = FEAT_DIR if os.access(FEAT_DIR, os.W_OK) else DIRS["cache"]      # Kaggle inputs are read-only
for b, s in CONFIGS:
    cache = os.path.join(FEAT_DIR, f"feat_{b}_{s}.npz")
    if os.path.exists(cache) and not SMOKE_TEST:
        try:   # v3.1 caches stored file names as an object array -> allow_pickle (the file is our own cache)
            z = np.load(cache, allow_pickle=True)
            if z["f"].shape[0] == len(X) and "files" in z and [str(v) for v in z["files"]] == [str(v) for v in meta.file]:
                FEATS[(b, s)] = (z["f"], z["f_flip"]); print(f"{b} @ {s}: cached features re-used"); continue
            print(f"{b} @ {s}: cached features do not match the current frames -> recomputing")
        except Exception as e:
            print(f"{b} @ {s}: cache unreadable ({e}) -> recomputing")
    t0 = time.time()
    m = build_extractor(b, s)
    f_ = make_fwd(m)
    Xs = resize_batch(X, s)
    f = np.concatenate([f_(tf.constant(Xs[i:i + 16])).numpy() for i in range(0, len(Xs), 16)])
    ff = np.concatenate([f_(tf.constant(Xs[i:i + 16, :, ::-1])).numpy() for i in range(0, len(Xs), 16)])
    np.savez_compressed(os.path.join(FEAT_OUT, os.path.basename(cache)), f=f, f_flip=ff, files=np.array([str(v) for v in meta.file], dtype=str)); FEATS[(b, s)] = (f, ff)
    print(f"{b} @ {s}: features {f.shape} in {time.time() - t0:.0f}s")
    keras.backend.clear_session()

# --- v4: non-CNN baseline gates (CPU): HOG, and image statistics (intensity histogram + Canny edge density + gradient grid) ---
from skimage.feature import hog
H_IMG, W_IMG = CONFIG["IMG_SIZE"]
def _gray(img):
    return cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
def f_hog(img):
    g = cv2.resize(_gray(img), (W_IMG // 2, H_IMG // 2), interpolation=cv2.INTER_AREA)
    return hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True).astype(np.float32)
def f_stats(img, gh=8, gw=4):
    g = _gray(img)
    hist = np.histogram(g, 32, (0, 256))[0] / g.size
    e = cv2.Canny(g, 50, 150) > 0
    mag = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    cells = [(slice(i * H_IMG // gh, (i + 1) * H_IMG // gh), slice(j * W_IMG // gw, (j + 1) * W_IMG // gw)) for i in range(gh) for j in range(gw)]
    return np.r_[hist, [e[c].mean() for c in cells], [mag[c].mean() / 255 for c in cells]].astype(np.float32)
HANDCRAFTED = {"HOG": f_hog, "ImgStats": f_stats}
HANDCRAFTED = {k: v for k, v in HANDCRAFTED.items() if k in CONFIG["HANDCRAFTED"]}
for name, fn in HANDCRAFTED.items():
    FEATS[(name, 1.0)] = (np.stack([fn(im) for im in X]), np.stack([fn(im[:, ::-1]) for im in X]))
    print(f"{name}: features {FEATS[(name, 1.0)][0].shape}")
ALL_CONFIGS = CONFIGS + [(n, 1.0) for n in HANDCRAFTED]

def extract(b, s, Xarr):
    # features of arbitrary frames (used for perturbed / augmented frames)
    if b in HANDCRAFTED:
        return np.stack([HANDCRAFTED[b](im) for im in Xarr])
    m = build_extractor(b, s); f_ = make_fwd(m); Xs = resize_batch(Xarr, s)
    out = np.concatenate([f_(tf.constant(Xs[i:i + 16])).numpy() for i in range(0, len(Xs), 16)])
    keras.backend.clear_session(); return out

## 5. Computational complexity
Parameters and FLOPs (1 multiply-add = 2 FLOPs) of each gate backbone at batch size 1, and of HED (OpenCV `getFLOPS`).

In [ ]:
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
def count_flops(model, shape):
    @tf.autograph.experimental.do_not_convert
    def fwd(x):
        return model(x, training=False)
    cf = tf.function(fwd).get_concrete_function(tf.TensorSpec([1, *shape], tf.float32))
    g = convert_variables_to_constants_v2(cf).graph
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation(); opts["output"] = "none"
    return tf.compat.v1.profiler.profile(graph=g, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts).total_float_ops

# --- HED (Caffe, OpenCV DNN) ---
class CropLayer(object):
    def __init__(self, params, blobs):
        self.ys = self.xs = self.h = self.w = 0
    def getMemoryShapes(self, inputs):
        i, t = inputs[0], inputs[1]
        self.ys, self.xs, self.h, self.w = int((i[2] - t[2]) / 2), int((i[3] - t[3]) / 2), t[2], t[3]
        return [[i[0], i[1], t[2], t[3]]]
    def forward(self, inputs):
        return [inputs[0][:, :, self.ys:self.ys + self.h, self.xs:self.xs + self.w]]
for p in (CONFIG["HED_PROTOTXT"], CONFIG["HED_MODEL"]):
    if not os.path.exists(p):
        raise FileNotFoundError(f"HED file missing: {p}")
try:
    cv2.dnn_unregisterLayer("Crop")
except Exception:
    pass
cv2.dnn_registerLayer("Crop", CropLayer)
def load_hed(cuda=False):
    net = cv2.dnn.readNetFromCaffe(CONFIG["HED_PROTOTXT"], CONFIG["HED_MODEL"])
    if cuda:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA); net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
    else:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV); net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    return net
hed = load_hed()
H, W = CONFIG["IMG_SIZE"]
def hed_forward(net, bgr):
    net.setInput(cv2.dnn.blobFromImage(bgr, 1.0, (bgr.shape[1], bgr.shape[0]), CONFIG["HED_MEAN"], swapRB=False, crop=False))
    return net.forward()
hed_flops = float(hed.getFLOPS([1, 3, H, W]))

rows = []
for b, s in CONFIGS:
    m = build_extractor(b, s)
    try:
        fl = count_flops(m, input_shape(s))
    except Exception as e:
        print("FLOP count failed for", b, s, e); fl = np.nan
    rows.append({"gate": b, "scale": s, "input": "x".join(map(str, input_shape(s)[:2])), "params_M": m.count_params() / 1e6,
                 "feature_dim": int(m.output_shape[-1]), "GFLOPs": fl / 1e9, "GFLOPs_rel_HED": fl / hed_flops})
    keras.backend.clear_session()
cx = pd.DataFrame(rows)
cx = pd.concat([cx, pd.DataFrame([{"gate": "HED (stage 2)", "scale": 1.0, "input": f"{H}x{W}", "params_M": np.nan,
                                    "feature_dim": np.nan, "GFLOPs": hed_flops / 1e9, "GFLOPs_rel_HED": 1.0}])], ignore_index=True)
save_table(cx.round(4), "complexity")
record("complexity", cx.to_dict(orient="records"))

## 6. Latency: CPU and GPU
Batch size 1 (one frame at a time, as in streaming), after warm-up. Each gate = backbone + linear head (the head adds a negligible dot product). GPU timings include the host-to-device copy and the result read-back. HED runs in OpenCV DNN on the CPU; OpenCV's CUDA backend is used only if the installed OpenCV has CUDA support (pip wheels do not).

In [ ]:
DEVICES = ["CPU"] + (["GPU"] if GPUS else [])
def time_calls(fn, arg, warmup, reps):
    for _ in range(warmup):
        fn(arg)
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(arg); ts.append(time.perf_counter() - t)
    return np.array(ts)

x1 = X[:1]
bgr1 = cv2.cvtColor(np.clip(X[0], 0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
def _measure_latency():
    # v4.1: latencies are measured in LAT_ROUNDS interleaved rounds (all models in each round), so that a temporary
    # slow-down of the shared cloud machine affects all models alike; mean_ms = median of the round means.
    R = CONFIG["LAT_ROUNDS"]
    calls = {}
    calls[("HED", 1.0, "CPU")] = (lambda a: hed_forward(hed, a), bgr1, CONFIG["HED_REPEATS"])
    keep = []
    for b, s in CONFIGS:
        xs = resize_batch(x1, s)
        for dev in DEVICES:
            with tf.device(f"/{dev}:0"):
                m = build_extractor(b, s); keep.append(m)
                w_head = np.random.randn(m.output_shape[-1]).astype(np.float32)
                fwd = make_fwd(m)
                def gate(a, fwd=fwd, w_head=w_head, dev=dev):
                    with tf.device(f"/{dev}:0"):
                        return float(fwd(tf.constant(a)).numpy()[0] @ w_head)    # .numpy() forces device sync
                gate(xs)
            calls[(b, s, dev)] = (gate, xs, CONFIG["LAT_REPEATS"])
    for name, fn in HANDCRAFTED.items():
        w_head = np.random.randn(FEATS[(name, 1.0)][0].shape[1]).astype(np.float32)
        calls[(name, 1.0, "CPU")] = (lambda a, fn=fn, w_head=w_head: float(fn(a) @ w_head), X[0], CONFIG["LAT_REPEATS"])
    times = {k: [] for k in calls}
    for r_ in range(R):
        for k, (fn, arg, n) in calls.items():
            times[k].append(time_calls(fn, arg, CONFIG["LAT_WARMUP"], n))
        print(f"latency round {r_ + 1}/{R} done")
    rows = []
    for (b, s, dev), ts in times.items():
        allt = np.concatenate(ts); rm = [t.mean() for t in ts]
        row = {"model": b, "scale": s, "device": dev, "mean_ms": 1e3 * float(np.median(rm)), "median_ms": 1e3 * float(np.median(allt)),
               "sd_ms": 1e3 * float(allt.std()), "round_means_ms": ";".join(f"{1e3 * v:.2f}" for v in rm)}
        rows.append(row)
        if b in HANDCRAFTED:
            for dev2 in DEVICES:
                if dev2 != "CPU":
                    rows.append({**row, "device": dev2, "note": "runs on CPU"})
    del keep; keras.backend.clear_session()
    return rows
lat_rows = cached("latency", _measure_latency)        # measured once; re-runs re-use the same latencies
L_HED = [r for r in lat_rows if r["model"] == "HED" and r["device"] == "CPU"][0]["mean_ms"] / 1e3
lat = pd.DataFrame(lat_rows)
lat["rho"] = lat.mean_ms / (1e3 * L_HED)          # gate cost relative to HED on the CPU
save_table(lat.round(4), "latency_cpu_gpu")
record("latency", lat.to_dict(orient="records")); record("L_HED_cpu_s", float(L_HED))

def rho_of(b, s, dev):
    r = lat[(lat.model == b) & (lat.scale == s) & (lat.device == dev)]
    return float(r.rho.iloc[0]) if len(r) else np.nan

fig, ax = plt.subplots(figsize=(7, 3.4))
SHORT = {"MobileNetV3Small": "MNv3-S", "MobileNetV3Large": "MNv3-L", "EfficientNetB0": "EffNet-B0", "ResNet50": "ResNet50", "HOG": "HOG", "ImgStats": "ImgStats"}
lab = [f"{SHORT.get(b, b)}\n{s:g}x" for b, s in CONFIGS]; xpos = np.arange(len(CONFIGS)); wd = 0.38
for k, (dev, col) in enumerate(zip(DEVICES, ["#2171b5", "#e6550d"])):
    vals = [lat[(lat.model == b) & (lat.scale == s) & (lat.device == dev)].mean_ms.iloc[0] for b, s in CONFIGS]
    ax.bar(xpos + (k - (len(DEVICES) - 1) / 2) * wd, vals, wd, color=col, label=f"gate on {dev}")
ax.axhline(1e3 * L_HED, color="#c0392b", ls="--", lw=1.2, label="HED on CPU")
ax.set_yscale("log"); ax.set_ylabel("Latency per frame (ms, log)"); ax.set_xticks(xpos); ax.set_xticklabels(lab, fontsize=8)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save_fig(fig, "fig_latency_cpu_gpu"); plt.show()

## 7. Gate heads and imbalance remedies
All heads produce a score where larger means *joint*.

| Strategy | Training data | Imbalance handling |
|---|---|---|
| LR | both classes | none (plain logistic regression) |
| LR-weighted | both classes | inverse-frequency class weights |
| LR-SMOTE | both classes | SMOTE oversampling of joints in feature space |
| Focal-weighted | both classes | focal loss (gamma = 2, alpha = 0.25) + class weights, i.e. the loss of the original gate |
| OneClass-Mahalanobis | background only (+ mirrored background) | Gaussian model of background features (PCA + Ledoit-Wolf), score = Mahalanobis distance |
| OneClass-kNN | background only (+ mirrored background) | mean cosine distance to the k nearest background frames |

In [ ]:
def _sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))

class FocalLogistic:
    # Linear model trained with the weighted binary focal loss of the paper (Eq. 4), L-BFGS
    def __init__(self, gamma, alpha, l2, class_weight):
        self.g, self.a, self.l2, self.cw = gamma, alpha, l2, class_weight
    def fit(self, Xf, yf):
        n, d = Xf.shape
        s = np.where(yf == 1, 1.0, -1.0)
        at = np.where(yf == 1, self.a, 1 - self.a) * np.where(yf == 1, self.cw[1], self.cw[0])
        def fg(theta):
            w, b = theta[:d], theta[d]
            z = Xf @ w + b
            pt = np.clip(_sigmoid(s * z), 1e-7, 1 - 1e-7)
            loss = np.sum(-at * (1 - pt) ** self.g * np.log(pt)) / n + self.l2 * w @ w
            dz = at * s * (1 - pt) ** self.g * (self.g * pt * np.log(pt) - (1 - pt)) / n
            return loss, np.r_[Xf.T @ dz + 2 * self.l2 * w, dz.sum()]
        r = minimize(fg, np.zeros(d + 1), jac=True, method="L-BFGS-B", options={"maxiter": 500})
        self.w, self.b = r.x[:d], r.x[d]
        return self
    def score(self, Xf):
        return _sigmoid(Xf @ self.w + self.b)

def smote(Xp, n_new, k, rng):
    if len(Xp) < 2 or n_new <= 0:
        return np.empty((0, Xp.shape[1]))
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xp))).fit(Xp)
    idx = nn.kneighbors(Xp, return_distance=False)[:, 1:]
    i = rng.integers(0, len(Xp), n_new); j = idx[i, rng.integers(0, idx.shape[1], n_new)]
    return Xp[i] + rng.random((n_new, 1)) * (Xp[j] - Xp[i])

class Head:
    def __init__(self, strategy, seed=0):
        self.st, self.seed = strategy, seed
    def fit(self, F, Fflip, yy, extra=None):
        # Order of operations (all inside the training fold): (1) add mirrored frames (and, if given, augmented copies),
        # (2) fit the standardizer, (3) SMOTE on the standardized joint features incl. mirrored joints, (4) fit the head.
        st = self.st
        one_class = st.startswith("OneClass")
        Ftr, ytr = F, yy
        if CONFIG["FLIP_AUG"] and Fflip is not None:          # mirrored frames (for one-class: mirrored background)
            Ftr, ytr = np.vstack([F, Fflip]), np.r_[yy, yy]
        for E in (extra or []):                                # v4: augmented copies (Section 19)
            Ftr, ytr = np.vstack([Ftr, E]), np.r_[ytr, yy]
        self.sc = StandardScaler().fit(Ftr[ytr == 0] if one_class else Ftr)
        Z = self.sc.transform(Ftr)
        n0, n1 = (ytr == 0).sum(), max((ytr == 1).sum(), 1)
        cw = {0: len(ytr) / (2 * n0), 1: len(ytr) / (2 * n1)}
        if st == "LR":
            self.m = LogisticRegression(C=CONFIG["C"], max_iter=5000).fit(Z, ytr)
        elif st == "LR-weighted":
            self.m = LogisticRegression(C=CONFIG["C"], max_iter=5000, class_weight="balanced").fit(Z, ytr)
        elif st == "LR-SMOTE":
            rng = np.random.default_rng(self.seed)
            Zs = smote(Z[ytr == 1], int(n0 - (ytr == 1).sum()), CONFIG["SMOTE_K"], rng)
            self.m = LogisticRegression(C=CONFIG["C"], max_iter=5000).fit(np.vstack([Z, Zs]), np.r_[ytr, np.ones(len(Zs), int)])
        elif st == "Focal-weighted":
            self.m = FocalLogistic(CONFIG["FOCAL_GAMMA"], CONFIG["FOCAL_ALPHA"], CONFIG["FOCAL_L2"], cw).fit(Z, ytr)
        elif st == "OneClass-Mahalanobis":
            Zn = Z[ytr == 0]
            self.pca = PCA(n_components=min(CONFIG["PCA_DIM"], len(Zn) - 1, Zn.shape[1]), random_state=0).fit(Zn)
            self.lw = LedoitWolf().fit(self.pca.transform(Zn))
        elif st == "OneClass-kNN":
            Zn = Z[ytr == 0]; Zn = Zn / (np.linalg.norm(Zn, axis=1, keepdims=True) + 1e-8)
            self.nn = NearestNeighbors(n_neighbors=min(CONFIG["KNN_K"], len(Zn)), metric="cosine").fit(Zn)
        return self
    def score(self, F):
        Z = self.sc.transform(F)
        if self.st in ("LR", "LR-weighted", "LR-SMOTE"):
            return self.m.predict_proba(Z)[:, 1]
        if self.st == "Focal-weighted":
            return self.m.score(Z)
        if self.st == "OneClass-Mahalanobis":
            return self.lw.mahalanobis(self.pca.transform(Z))
        Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
        return self.nn.kneighbors(Zn)[0].mean(axis=1)

## 8. Recall-constrained threshold selection and grouped cross-validation
For a recall target r*, the threshold is chosen **inside the training fold**: inner grouped cross-validation gives out-of-fold scores, and tau is the largest threshold that keeps inner recall >= r* (placed midway to the next lower background score for a margin). The model is then refit on the whole training fold and applied to the test fold.

For each configuration we report, pooled over the test folds of one repetition (mean +- SD over repetitions):
- **PR-AUC**, ROC-AUC, and the threshold-free **FPR at 100% recall** (the share of background frames scoring at least as high as the lowest joint),
- with the selected tau: achieved **recall**, **FPR**, the share **q** of frames routed to HED, and the predicted speed-up **S = 1/(rho + mean q)** for the gate on CPU and on GPU (v4: the speed-up of the mean per-frame cost, i.e. computed from the mean of q over repetitions, not the mean of per-repetition speed-ups; its SD follows from the SD of q by the delta method),
- v4: PR-AUC and FPR@R100 also **per fold** (mean +- SD over the 15 folds); all per-fold numbers are saved in `tables/per_fold_results.csv`.
- for supervised heads also the fixed thresholds 0.5 and the legacy 0.35.

In [ ]:
def select_tau(scores, labels, r_target):
    pos = np.sort(scores[labels == 1])
    if len(pos) == 0:
        return np.inf
    k = int(np.floor((1 - r_target) * len(pos) + 1e-9))
    t_pos = pos[min(k, len(pos) - 1)]
    lower = scores[(labels == 0) & (scores < t_pos)]
    return 0.5 * (t_pos + lower.max()) if len(lower) else t_pos

def fpr_at_recall(scores, labels, r=1.0):
    pos = np.sort(scores[labels == 1])
    if len(pos) == 0:
        return np.nan
    t = pos[int(np.floor((1 - r) * len(pos) + 1e-9))]
    return float((scores[labels == 0] >= t).mean())

def splitter(kind, n_splits, seed):
    if kind == "group":
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

def _sub(A, rows):
    return None if A is None else A[rows]

def inner_taus(tr, F, Ff, strat, seed, kind="group", extra=None, r_targets=(1.0,), n_inner=None):
    # threshold(s) from inner cross-validation on the training rows `tr` only
    inner = splitter("frame" if kind == "frame" else "group", n_inner or CONFIG["INNER_FOLDS"], seed + 100)
    sc = np.full(len(tr), np.nan)
    try:
        for itr, ite in inner.split(tr, y[tr], groups[tr]):
            a, b_ = tr[itr], tr[ite]
            if y[a].sum() == 0:
                continue
            h = Head(strat, seed).fit(F[a], _sub(Ff, a), y[a], None if extra is None else [E[a] for E in extra])
            sc[ite] = h.score(F[b_])
    except ValueError:
        pass
    ok = ~np.isnan(sc)
    taus = {r: (select_tau(sc[ok], y[tr][ok], r) if ok.any() else np.nan) for r in r_targets}
    taus["min_pos"] = float(sc[ok][y[tr][ok] == 1].min()) if (ok & (y[tr] == 1)).any() else np.nan
    return taus

def run_cv_full(b, s, strat, seed, kind="group", outer=None, idx=None, test_sets=None, extra=None,
                n_outer=None, n_inner=None, r_targets=None, feats=None):
    # Cross-validation restricted to rows `idx` (default: all). Returns out-of-fold scores, thresholds and fold ids.
    F, Ff = feats if feats is not None else FEATS[(b, s)]
    idx = np.arange(len(y)) if idx is None else np.asarray(idx)
    r_targets = list(r_targets or CONFIG["RECALL_TARGETS"])
    oof, oof_flip, fold = np.full(len(y), np.nan), np.full(len(y), np.nan), np.full(len(y), -1)
    tests = {k: np.full(len(y), np.nan) for k in (test_sets or {})}
    taus = {r: np.full(len(y), np.nan) for r in r_targets + ["min_pos"]}
    if isinstance(outer, LeaveOneGroupOut):
        it = outer.split(idx, y[idx], stations[idx])
    else:
        it = (outer or splitter(kind, n_outer or CONFIG["N_FOLDS"], seed)).split(idx, y[idx], groups[idx])
    for k, (a, b_) in enumerate(it):
        tr, te = idx[a], idx[b_]
        if y[tr].sum() == 0:
            continue
        t_ = inner_taus(tr, F, Ff, strat, seed, kind, extra, r_targets, n_inner)
        h = Head(strat, seed).fit(F[tr], _sub(Ff, tr), y[tr], None if extra is None else [E[tr] for E in extra])
        oof[te] = h.score(F[te]); fold[te] = k
        if Ff is not None:
            oof_flip[te] = h.score(Ff[te])
        for nm, T in (test_sets or {}).items():
            tests[nm][te] = h.score(T[te])
        for r in taus:
            taus[r][te] = t_[r]
    return {"oof": oof, "flip": oof_flip, "taus": taus, "fold": fold, "tests": tests, "idx": idx}

def run_cv(b, s, strat, seed, kind="group", outer=None):
    R = run_cv_full(b, s, strat, seed, kind, outer)
    return R["oof"], R["flip"], {r: R["taus"][r] for r in CONFIG["RECALL_TARGETS"]}

def summarize(oof, oof_flip, taus, b, s, strat, mask=None):
    m = ~np.isnan(oof) if mask is None else (mask & ~np.isnan(oof))
    yy, sc, scf = y[m], oof[m], oof_flip[m]
    out = {"pr_auc": average_precision_score(yy, sc) if 0 < yy.sum() < len(yy) else np.nan,
           "roc_auc": roc_auc_score(yy, sc) if 0 < yy.sum() < len(yy) else np.nan,
           "fpr_at_r100": fpr_at_recall(sc, yy, 1.0), "fpr_at_r95": fpr_at_recall(sc, yy, 0.95)}
    def dec_stats(dec, tag):
        tp, fp = int((dec & (yy == 1)).sum()), int((dec & (yy == 0)).sum())
        out[f"recall_{tag}"] = tp / max(yy.sum(), 1); out[f"fpr_{tag}"] = fp / max((yy == 0).sum(), 1)
        out[f"missed_{tag}"] = int(yy.sum() - tp); q = dec.mean(); out[f"q_{tag}"] = q
        for dev in DEVICES:
            out[f"S_{dev}_{tag}"] = 1 / (rho_of(b, s, dev) + q)
    for r, t in taus.items():
        if r == "min_pos":
            continue
        dec = sc >= t[m]
        dec_stats(dec, f"r{int(r * 100)}")
        if r == 1.0 and not np.all(np.isnan(scf)):
            out["recall_mirror_r100"] = float(((scf >= t[m]) & (yy == 1)).sum() / max(yy.sum(), 1))
            out["fpr_mirror_r100"] = float(((scf >= t[m]) & (yy == 0)).sum() / max((yy == 0).sum(), 1))
    if not strat.startswith("OneClass"):
        dec_stats(sc >= 0.5, "t050"); dec_stats(sc >= CONFIG["LEGACY_TAU"], "t035")
    return out

def perfold(R, b, s, strat, seed, proto="group"):
    # per-fold metrics (released as per_fold_results.csv; PR-AUC averaged over folds for Reviewer 1)
    rows = []
    for k in sorted(set(R["fold"][R["fold"] >= 0])):
        m = R["fold"] == k; yy, sc = y[m], R["oof"][m]; t = R["taus"][1.0][m][0]; dec = sc >= t
        rows.append({"gate": b, "scale": s, "strategy": strat, "seed": seed, "protocol": proto, "fold": int(k),
                     "n_pos": int(yy.sum()), "n_neg": int((yy == 0).sum()),
                     "pr_auc": average_precision_score(yy, sc) if 0 < yy.sum() < len(yy) else np.nan,
                     "fpr_at_r100": fpr_at_recall(sc, yy, 1.0) if yy.sum() and (yy == 0).any() else np.nan,
                     "tau": float(t), "recall": float(dec[yy == 1].mean()) if yy.sum() else np.nan,
                     "fpr": float(dec[yy == 0].mean()) if (yy == 0).any() else np.nan, "q": float(dec.mean()),
                     "missed": int((~dec & (yy == 1)).sum())})
    return rows

def speedups_from_mean_q(dm, dsd=None):
    # S = 1/(rho + mean q): the speed-up of the mean per-frame cost (Reviewer 1, comments 9-10).
    # SD of S by the delta method from the SD of q across repetitions.
    rho = {dev: np.array([rho_of(b, s, dev) for b, s, _ in dm.index]) for dev in DEVICES}
    for c in [c for c in dm.columns if c.startswith("q_")]:
        tag = c[2:]
        for dev in DEVICES:
            den = rho[dev] + dm[c].values
            dm[f"S_{dev}_{tag}"] = 1 / den
            if dsd is not None:
                dsd[f"S_{dev}_{tag}"] = dsd[c].values / den ** 2
    return dm, dsd

In [ ]:
def _main_cv():
    rows, oofs, pf = [], {}, []
    t0 = time.time()
    for (b, s), strat in itertools.product(ALL_CONFIGS, CONFIG["STRATEGIES"]):
        for seed in CONFIG["CV_SEEDS"]:
            R = run_cv_full(b, s, strat, seed, "group")
            taus = {r: R["taus"][r] for r in CONFIG["RECALL_TARGETS"]}
            oofs[(b, s, strat, seed)] = (R["oof"], R["flip"], taus)
            pf += perfold(R, b, s, strat, seed)
            rows.append({"gate": b, "scale": s, "strategy": strat, "seed": seed, **summarize(R["oof"], R["flip"], taus, b, s, strat)})
        print(f"{b}@{s} {strat}: done ({time.time() - t0:.0f}s)")
    return rows, oofs, pf
cv_rows, OOF, PERFOLD = cached("main_cv", _main_cv)
cv = pd.DataFrame(cv_rows)
cv.to_csv(os.path.join(DIRS["tables"], "cv_grouped_per_seed.csv"), index=False)
pf_df = pd.DataFrame(PERFOLD)
pf_df.to_csv(os.path.join(DIRS["tables"], "per_fold_results.csv"), index=False)
num = [c for c in cv.columns if c not in ("gate", "scale", "strategy", "seed")]
cv_mean = cv.groupby(["gate", "scale", "strategy"])[num].mean()
cv_sd = cv.groupby(["gate", "scale", "strategy"])[num].std().fillna(0)
cv_mean, cv_sd = speedups_from_mean_q(cv_mean, cv_sd)
fold_auc = pf_df.groupby(["gate", "scale", "strategy"]).pr_auc.agg(["mean", "std"])
cv_mean["pr_auc_fold"] = fold_auc["mean"]; cv_sd["pr_auc_fold"] = fold_auc["std"]
fold_fpr = pf_df.groupby(["gate", "scale", "strategy"]).fpr_at_r100.agg(["mean", "std"])
cv_mean["fpr_at_r100_fold"] = fold_fpr["mean"]; cv_sd["fpr_at_r100_fold"] = fold_fpr["std"]
cv_mean.round(4).to_csv(os.path.join(DIRS["tables"], "cv_grouped_mean.csv"))
cv_sd.round(4).to_csv(os.path.join(DIRS["tables"], "cv_grouped_sd.csv"))
show_cols = ["pr_auc", "pr_auc_fold", "fpr_at_r100", "recall_r100", "fpr_r100", "q_r100", "S_CPU_r100"] + (["S_GPU_r100"] if "GPU" in DEVICES else [])
display(cv_mean[show_cols].round(3).sort_values("S_CPU_r100", ascending=False))

In [ ]:
# --- 8.1 Best configurations -----------------------------------------------------------------
# CPU-best: highest predicted CPU speed-up among configurations with 100% recall in EVERY repetition
# (falls back to ranking by rounded recall if none qualifies). GPU-best: same constraint, ranked by GPU speed-up.
rank = cv_mean.assign(rec_round=cv_mean.recall_r100.round(2)).sort_values(["rec_round", "S_CPU_r100"], ascending=False)
strict = cv_mean[cv_mean.recall_r100 >= 0.9999]
BEST = strict.S_CPU_r100.idxmax() if len(strict) else rank.index[0]
GBEST = strict.S_GPU_r100.idxmax() if (len(strict) and "S_GPU_r100" in strict.columns) else None
strict_sup = strict[[not i[2].startswith("OneClass") for i in strict.index]]
ACC = strict_sup.sort_values(["fpr_r100", "S_GPU_r100" if "S_GPU_r100" in strict_sup else "S_CPU_r100"], ascending=[True, False]).index[0] if len(strict_sup) else None
print("GPU-best configuration:", GBEST, "| most accurate (lowest FPR at 100% recall):", ACC)
best_by_strategy = rank.reset_index().groupby("strategy").head(1).set_index(["gate", "scale", "strategy"])
best_by_gate = rank.reset_index().groupby(["gate", "scale"]).head(1).set_index(["gate", "scale", "strategy"])
REF = ("ResNet50", 1.0, "Focal-weighted") if ("ResNet50", 1.0, "Focal-weighted") in cv_mean.index else rank.index[-1]
print("Best configuration:", BEST, "| reference (original design):", REF)
fmt = lambda m, sd: f"{m:.3f} ± {sd:.3f}"
tbl = []
for idx in list(best_by_gate.index):
    m, sd = cv_mean.loc[idx], cv_sd.loc[idx]
    tbl.append({"gate": idx[0], "scale": idx[1], "best strategy": idx[2],
                **{c: fmt(m[c], sd[c]) for c in show_cols}})
best_tbl = pd.DataFrame(tbl)
save_table(best_tbl, "best_per_gate")
strat_tbl = pd.DataFrame([{"strategy": s_, **{c: fmt(cv_mean.loc[(BEST[0], BEST[1], s_)][c], cv_sd.loc[(BEST[0], BEST[1], s_)][c])
                                                  for c in show_cols + (["recall_t050", "fpr_t050", "recall_t035", "fpr_t035"])
                                                  if c in cv_mean.columns and not np.isnan(cv_mean.loc[(BEST[0], BEST[1], s_)][c])}}
                         for s_ in CONFIG["STRATEGIES"]])
save_table(strat_tbl, "strategies_on_best_gate")
record("best", list(BEST)); record("reference", list(REF)); record("gpu_best", list(GBEST) if GBEST else None)
record("most_accurate", list(ACC) if ACC else None)
record("cv_mean", cv_mean.reset_index().to_dict(orient="records"))

## 9. Figures: design space, score scales, PR curves

In [ ]:
# --- Pareto: gate cost (rho) vs routed fraction q at r* = 100%, with iso-speed-up curves ---
fig, ax = plt.subplots(figsize=(7.6, 4.2))
x_lim = min(1.0, lat[(lat.device == "CPU") & (lat.model != "HED")].rho.max() * 1.25 + 0.02)
markers = {"LR": "o", "LR-weighted": "s", "LR-SMOTE": "D", "Focal-weighted": "^", "OneClass-Mahalanobis": "P", "OneClass-kNN": "X"}
cols = dict(zip(CONFIG["BACKBONES"] + list(HANDCRAFTED), ["#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#666666", "#a6761d"]))
rr = np.linspace(0.001, 0.8, 300)
for S_ in [2, 3, 5, 8]:
    qq = 1 / S_ - rr; ok = qq > 0
    ax.plot(rr[ok], qq[ok], color="#bbbbbb", lw=0.8, ls="--")
    if 1 / S_ <= x_lim:
        ax.text(1 / S_, 0.006, f"S={S_}", fontsize=7, color="#777777", ha="center")
    else:
        ax.text(x_lim * 0.98, 1 / S_ - x_lim * 0.98 + 0.01, f"S={S_}", fontsize=7, color="#777777", ha="right")
for (b, s, st_), m in cv_mean.iterrows():
    ax.scatter(rho_of(b, s, "CPU"), m.q_r100, marker=markers[st_], color=cols[b], s=40 if s == 1.0 else 18,
               edgecolor="black" if (b, s, st_) == BEST else "none", lw=1.2)
ax.axhline(PI, color="#555555", lw=0.8, ls=":"); ax.text(0.01, PI, r"$q=\pi$ (perfect gate)", ha="left", va="bottom", fontsize=7)
from matplotlib.lines import Line2D
h1 = [Line2D([], [], color=cols[b], marker="o", ls="", label=b) for b in CONFIG["BACKBONES"] + list(HANDCRAFTED)]
h2 = [Line2D([], [], color="#555555", marker=mk, ls="", label=st_) for st_, mk in markers.items()]
ax.legend(handles=h1 + h2, frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_xlabel(r"Gate cost $\rho = L_{gate}/L_{HED}$ (CPU)"); ax.set_ylabel(r"Frames routed to HED, $q$ (recall target 100%)")
ax.set_xlim(0, x_lim); ax.set_ylim(0, min(1, cv_mean.q_r100.max() * 1.1 + 0.02))
fig.tight_layout(); save_fig(fig, "fig_design_space"); plt.show()

# --- score scale: the same gate trained with different remedies (repeat 0) ---
b, s, _ = BEST
sel = [st_ for st_ in ["LR", "LR-weighted", "Focal-weighted"] if (b, s, st_, CONFIG["CV_SEEDS"][0]) in OOF]
fig, axes = plt.subplots(1, len(sel), figsize=(3.3 * len(sel), 2.8), sharey=True)
for ax, st_ in zip(np.atleast_1d(axes), sel):
    oof, _, taus = OOF[(b, s, st_, CONFIG["CV_SEEDS"][0])]
    bins = np.linspace(0, 1, 26)
    ax.hist(oof[y == 0], bins=bins, color="#9ecae1", label="background", density=True)
    ax.hist(oof[y == 1], bins=bins, color="#e6550d", alpha=0.8, label="joint", density=True)
    ax.axvline(np.nanmedian(taus[1.0]), color="black", lw=1.2, label=r"selected $\tau$ (median)")
    ax.axvline(CONFIG["LEGACY_TAU"], color="#777777", lw=1, ls=":", label=r"fixed $\tau$=0.35")
    ax.set_title(st_, fontsize=9); ax.set_xlabel("gate score")
np.atleast_1d(axes)[0].set_ylabel("density"); np.atleast_1d(axes)[-1].legend(frameon=False, fontsize=7)
fig.tight_layout(); save_fig(fig, "fig_score_distributions"); plt.show()

# --- PR curves of the best head per backbone (scale 1.0, repeat 0) ---
fig, ax = plt.subplots(figsize=(4.6, 3.6))
for (b_, s_, st_) in best_by_gate.index:
    if s_ != 1.0:
        continue
    oof = OOF[(b_, s_, st_, CONFIG["CV_SEEDS"][0])][0]
    p, r, _ = precision_recall_curve(y, oof)
    ax.plot(r, p, color=cols[b_], lw=1.5, label=f"{b_} ({st_}), AP={average_precision_score(y, oof):.2f}")
ax.axhline(PI, color="#777777", ls="--", lw=1)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_ylim(0, 1.05); ax.legend(frameon=False, fontsize=6.5, loc="lower left")
fig.tight_layout(); save_fig(fig, "fig_pr_curves_grouped_cv"); plt.show()

## 10. Robustness: leakage, unseen stations, mirrored frames
* **Leakage:** the best configuration evaluated with ordinary (frame-level) stratified CV versus recording-grouped CV. The difference shows how optimistic a random frame split is.
* **Unseen stations:** leave-one-station-out (the station is the first field of the file name); per-station recall and FPR are saved in `tables/per_station_logo.csv`.
* **Mirrored frames:** test folds mirrored horizontally, thresholds unchanged, heads trained with and without flip augmentation.

In [ ]:
rob, pst_rows = [], {}
for cfg_ in dict.fromkeys([BEST, REF]):
    b, s, st_ = cfg_
    tag = "best" if cfg_ == BEST else "ref"
    # out-of-fold predictions (repeat 0, recording-grouped CV) for traceability
    oof0, _, taus0 = OOF[(b, s, st_, CONFIG["CV_SEEDS"][0])]
    pd.DataFrame({"file": meta.file, "label": y, "station": meta.station, "recording": meta.recording,
                  "score": oof0, "tau": taus0[1.0], "routed": (oof0 >= taus0[1.0]).astype(int)}).to_csv(
        os.path.join(DIRS["tables"], f"oof_predictions_{tag}.csv"), index=False)
    for kind in ["frame", "group"]:
        res = [summarize(*run_cv(b, s, st_, sd, kind), b, s, st_) for sd in CONFIG["CV_SEEDS"]]
        rob.append({"config": f"{b}@{s} {st_}", "gate": b, "scale": s, "protocol": f"{kind}-level 5-fold", **pd.DataFrame(res).mean().to_dict()})
    if meta.station.nunique() > 1:
        oof_l, oofl_flip, taus_l = run_cv(b, s, st_, CONFIG["CV_SEEDS"][0], "group", outer=LeaveOneGroupOut())
        res = summarize(oof_l, oofl_flip, taus_l, b, s, st_)
        rob.append({"config": f"{b}@{s} {st_}", "gate": b, "scale": s, "protocol": "leave-one-station-out", **res})
        dec = oof_l >= taus_l[1.0]
        for stn in sorted(meta.station.unique()):
            m_ = stations == stn; yy, dd = y[m_], dec[m_]
            r_ = pst_rows.setdefault(stn, {"station": stn, "frames": int(m_.sum()), "joints": int(yy.sum())})
            r_[f"recall_{tag}"] = float(dd[yy == 1].mean()) if yy.sum() else np.nan
            r_[f"fpr_{tag}"] = float(dd[yy == 0].mean()) if (yy == 0).sum() else np.nan
rob_df = pd.DataFrame(rob)
for dev in DEVICES:        # v4: speed-up of the mean cost (consistent with the reported q)
    rob_df[f"S_{dev}_r100"] = [1 / (rho_of(r.gate, r.scale, dev) + r.q_r100) for r in rob_df.itertuples()]
if pst_rows:
    save_table(pd.DataFrame(list(pst_rows.values())).round(4), "per_station_logo")

# --- station/label confound checks ---------------------------------------------------------
# (a) performance restricted to stations that contain both joints and background frames;
# (b) a "station-only" baseline whose score is the joint rate of the frame's station in the training folds.
mixed = sorted(s_ for s_ in meta.station.unique() if 0 < y[stations == s_].sum() < (stations == s_).sum())
mm = np.isin(stations, mixed)
conf_rows = []
def _conf(name, subset, score, mask, dec=None):
    yy, sc = y[mask], score[mask]
    row = {"model": name, "subset": subset, "frames": int(mask.sum()), "joints": int(yy.sum()),
           "pr_auc": average_precision_score(yy, sc) if 0 < yy.sum() < len(yy) else np.nan,
           "fpr_at_r100": fpr_at_recall(sc, yy, 1.0)}
    if dec is not None:
        row["recall"] = float(dec[mask & (y == 1)].mean()) if (mask & (y == 1)).any() else np.nan
        row["fpr"] = float(dec[mask & (y == 0)].mean()) if (mask & (y == 0)).any() else np.nan
    conf_rows.append(row)
for cfg_ in dict.fromkeys([BEST, REF]):
    oof0, _, taus0 = OOF[(cfg_[0], cfg_[1], cfg_[2], CONFIG["CV_SEEDS"][0])]
    name = f"{cfg_[0]}@{cfg_[1]} {cfg_[2]}"
    _conf(name, "all frames", oof0, np.ones(len(y), bool), oof0 >= taus0[1.0])
    if mm.any():
        _conf(name, "stations with both classes", oof0, mm, oof0 >= taus0[1.0])
st_score = np.zeros(len(y))
for tr, te in splitter("group", CONFIG["N_FOLDS"], CONFIG["CV_SEEDS"][0]).split(np.zeros(len(y)), y, groups):
    rates = pd.Series(y[tr]).groupby(stations[tr]).mean()
    st_score[te] = [rates.get(s_, y[tr].mean()) for s_ in stations[te]]
st_score += 1e-6 * np.random.default_rng(0).random(len(y))      # break ties at random
_conf("station-only baseline", "all frames", st_score, np.ones(len(y), bool))
if mm.any():
    _conf("station-only baseline", "stations with both classes", st_score, mm)
conf_df = pd.DataFrame(conf_rows)
save_table(conf_df.round(4), "confound_check")
record("mixed_stations", mixed); record("confound_check", conf_df.to_dict(orient="records"))
save_table(rob_df[["config", "protocol", "pr_auc", "fpr_at_r100", "recall_r100", "fpr_r100", "q_r100", "S_CPU_r100"]].round(3), "robustness_protocols")

mir = []
flip_setting = CONFIG["FLIP_AUG"]
sup = [i for i in rank.index if not i[2].startswith("OneClass")]
BEST_SUP = sup[0] if sup else None
record("best_supervised", list(BEST_SUP) if BEST_SUP else None)
for cfg_ in dict.fromkeys([c for c in (BEST, BEST_SUP) if c]):
    b, s, st_ = cfg_
    for fa in [True, False]:
        CONFIG["FLIP_AUG"] = fa
        res = pd.DataFrame([summarize(*run_cv(b, s, st_, sd, "group"), b, s, st_) for sd in CONFIG["CV_SEEDS"]]).mean()
        mir.append({"config": f"{b}@{s} {st_}", "flip_augmentation": fa, "recall_r100": res.recall_r100, "fpr_r100": res.fpr_r100,
                    "recall_mirrored": res.recall_mirror_r100, "fpr_mirrored": res.fpr_mirror_r100})
CONFIG["FLIP_AUG"] = flip_setting
if mir:
    save_table(pd.DataFrame(mir).round(3), "mirror_stress_test")
record("robustness", rob_df.to_dict(orient="records")); record("mirror", mir)

## 11. End-to-end validation
Moved to Section 22 (three deployment settings, all repetitions, stream replay).

## 12. Legacy fine-tuned ResNet50 gate: effect of re-selecting the threshold
Uses the per-frame scores saved by `MLWA_publication_ready_notebook_v2` (original split). The threshold is re-selected on the validation scores with the same rule and applied to the test frames.

In [ ]:
vp, tp_ = (os.path.join(CONFIG["LEGACY_V2_TABLES"], f) for f in ("validation_predictions.csv", "test_predictions.csv"))
legacy = None
if os.path.exists(vp) and os.path.exists(tp_):
    v, t = pd.read_csv(vp), pd.read_csv(tp_)
    tau_star = select_tau(v.prob.values, v.label.values, 1.0)
    rows = []
    for name, tau in [("fixed 0.35", CONFIG["LEGACY_TAU"]), ("fixed 0.5", 0.5), ("validation-selected", tau_star)]:
        for split, d in [("validation", v), ("test", t)]:
            dec = d.prob.values >= tau; yy = d.label.values
            rows.append({"threshold": name, "tau": round(float(tau), 4), "split": split,
                         "recall": (dec & (yy == 1)).sum() / max(yy.sum(), 1), "FP": int((dec & (yy == 0)).sum()),
                         "fpr": (dec & (yy == 0)).sum() / (yy == 0).sum(), "q": dec.mean()})
    legacy = pd.DataFrame(rows)
    save_table(legacy.round(4), "legacy_gate_threshold")
    record("legacy_threshold", legacy.to_dict(orient="records"))
else:
    print("v2 prediction tables not found - section skipped.")

## 13. Error analysis: hardest frames of the best configuration
Background frames with the highest scores and joint frames with the lowest scores (out-of-fold, repeat 0).

In [ ]:
b, s, st_ = BEST
oof, _, taus = OOF[(b, s, st_, CONFIG["CV_SEEDS"][0])]
neg_i = np.argsort(-np.where(y == 0, oof, -np.inf))[:4]; pos_i = np.argsort(np.where(y == 1, oof, np.inf))[:4]
fig, axes = plt.subplots(2, 4, figsize=(13, 4.4))
for ax, i in zip(axes.ravel(), list(neg_i) + list(pos_i)):
    ax.imshow(cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)); ax.axis("off")
    routed = oof[i] >= taus[1.0][i]
    ax.set_title(f"{meta.file[i]}\n{'joint' if y[i] else 'background'} | score {oof[i]:.2f} | {'to HED' if routed else 'dropped'}",
                 fontsize=7, color="#b03a2e" if routed != bool(y[i]) else "#1b5e20")
fig.tight_layout(); save_fig(fig, "fig_hardest_frames"); plt.show()

## 15. Uncertainty and statistical comparisons (Reviewer 1.6, Reviewer 2.10, Reviewer 3)
* **Per-fold metrics:** PR-AUC and FPR@R100 computed inside each test fold and averaged (Section 8, `pr_auc_fold`).
* **Recall bounds:** with k of n joints detected, the exact Clopper-Pearson 95% interval and the one-sided 95% lower bound. Repeated cross-validation re-uses the same 29 joints, so the bound uses n = 29, not 3 x 29.
* **Recording-cluster bootstrap:** recordings are resampled with replacement (frames of a recording stay together); the pooled out-of-fold predictions of each repetition are re-weighted, the metric is averaged over repetitions, and the 2.5/97.5 percentiles give a 95% CI.
* **Paired tests:** corrected resampled t-test of Nadeau & Bengio (2003) on the per-fold differences over the 5 folds x 3 repetitions, with Holm correction within each family; Friedman test across the supervised heads on the best backbone.

In [ ]:
from scipy.stats import t as t_dist, friedmanchisquare
SEL = {"best": BEST, "gpu_best": GBEST, "reference": REF}
SEL = {k: v for k, v in SEL.items() if v is not None}
import hashlib
SIGN = hashlib.md5(repr(sorted((k, (v[0], float(v[1]), v[2])) for k, v in SEL.items())).encode()).hexdigest()[:6]
print("selected gates:", SEL, "| cache signature:", SIGN)
def cfg_str(c):
    return f"{c[0]}@{c[1]} {c[2]}"

# --- recall bounds ---
rb = []
for tag, c in SEL.items():
    for seed in CONFIG["CV_SEEDS"]:
        oof, _, taus = OOF[(*c, seed)]
        k = int(((oof >= taus[1.0]) & (y == 1)).sum()); n = int(y.sum())
        lo, hi = clopper_pearson(k, n)
        rb.append({"config": cfg_str(c), "role": tag, "seed": seed, "detected": k, "joints": n, "recall": k / n,
                   "cp_low": lo, "cp_high": hi, "one_sided_95_lower": float(beta.ppf(0.05, k, n - k + 1)) if k > 0 else 0.0})
rb_df = pd.DataFrame(rb); save_table(rb_df.round(4), "recall_bounds")

# --- recording-cluster bootstrap CIs ---
def boot_ci(c, B, seed=0):
    rng = np.random.default_rng(seed)
    recs = np.unique(groups); rec_idx = {r: np.where(groups == r)[0] for r in recs}
    runs = [OOF[(*c, sd)] for sd in CONFIG["CV_SEEDS"]]
    stats = []
    for _ in range(B):
        draw = rng.choice(recs, len(recs), replace=True)
        w = np.zeros(len(y))
        for r in draw:
            w[rec_idx[r]] += 1
        per = []
        for oof, _, taus in runs:
            dec = oof >= taus[1.0]; pw, nw = w * (y == 1), w * (y == 0)
            if pw.sum() == 0 or nw.sum() == 0:
                continue
            q = (w * dec).sum() / w.sum()
            per.append({"recall": (pw * dec).sum() / pw.sum(), "fpr": (nw * dec).sum() / nw.sum(), "q": q,
                        "pr_auc": average_precision_score(y, oof, sample_weight=w),
                        **{f"S_{dev}": 1 / (rho_of(c[0], c[1], dev) + q) for dev in DEVICES}})
        if per:
            stats.append(pd.DataFrame(per).mean())
    S_ = pd.DataFrame(stats)
    return S_.quantile(0.025), S_.quantile(0.975)
def _boot():
    rows = []
    for tag, c in SEL.items():
        lo, hi = boot_ci(c, CONFIG["BOOTSTRAP"])
        for k_ in lo.index:
            point = cv_mean.loc[c][{"recall": "recall_r100", "fpr": "fpr_r100", "q": "q_r100", "pr_auc": "pr_auc"}.get(k_, f"{k_}_r100")]
            rows.append({"config": cfg_str(c), "role": tag, "metric": k_, "point": point, "ci_low": lo[k_], "ci_high": hi[k_]})
    return pd.DataFrame(rows)
boot_df = cached(f"bootstrap_{SIGN}", _boot); save_table(boot_df.round(4), "bootstrap_ci")

# --- corrected resampled t-test (Nadeau & Bengio 2003) on per-fold metrics ---
n_te = len(y) / CONFIG["N_FOLDS"]; n_tr = len(y) - n_te
def corrected_t(d):
    d = np.asarray(d, float); d = d[~np.isnan(d)]; J = len(d)
    if J < 2 or np.var(d, ddof=1) == 0:
        return np.nan, 1.0 if np.all(d == 0) else np.nan
    t_ = d.mean() / np.sqrt((1 / J + n_te / n_tr) * np.var(d, ddof=1))
    return float(t_), float(2 * t_dist.sf(abs(t_), J - 1))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); m = len(p); run = 0
    for r, i in enumerate(o):
        run = max(run, min(1, (m - r) * p[i])); adj[i] = run
    return adj
def pf_metric(c, metric):
    d = pf_df[(pf_df.gate == c[0]) & (pf_df.scale == c[1]) & (pf_df.strategy == c[2])].sort_values(["seed", "fold"])
    return d[metric].values
tests = []
heads = [st_ for st_ in CONFIG["STRATEGIES"] if not st_.startswith("OneClass")]
fam = {"heads on best backbone": [((BEST[0], BEST[1], a), (BEST[0], BEST[1], b_)) for a, b_ in itertools.combinations(heads, 2)],
       "selected gates": [(a, b_) for a, b_ in itertools.combinations(list(dict.fromkeys(SEL.values())), 2)]}
for fname, pairs in fam.items():
    block = []
    for a, b_ in pairs:
        for metric in ["pr_auc", "fpr_at_r100", "q"]:
            d = pf_metric(a, metric) - pf_metric(b_, metric)
            t_, p_ = corrected_t(d)
            block.append({"family": fname, "A": cfg_str(a), "B": cfg_str(b_), "metric": metric, "mean_diff": np.nanmean(d), "t": t_, "p": p_})
    if block:
        ps = holm([r["p"] if not np.isnan(r["p"]) else 1.0 for r in block])
        for r, pa in zip(block, ps):
            r["p_holm"] = pa
        tests += block
tests_df = pd.DataFrame(tests); save_table(tests_df.round(4), "significance_tests")
fr = [pf_metric((BEST[0], BEST[1], st_), "pr_auc") for st_ in heads]
fr_identical = bool(all(np.allclose(fr[0], f_, equal_nan=True) for f_ in fr[1:]))
fr_stat, fr_p = (np.nan, np.nan) if fr_identical or len(fr) < 3 else friedmanchisquare(*fr)
record("friedman_heads_best_backbone", {"chi2": float(fr_stat), "p": float(fr_p), "identical": fr_identical})
if fr_identical:
    print("Per-fold PR-AUC is identical for all supervised heads in every fold -> Friedman test not defined (no differences).")
print(f"Friedman test, supervised heads on {BEST[0]}@{BEST[1]} (per-fold PR-AUC): chi2 = {fr_stat:.2f}, p = {fr_p:.3f}")

## 16. Nested evaluation of the complete selection procedure (Reviewer 3; Cawley & Talbot, 2010)
The configuration reported in Sections 8-9 is chosen from all candidates using the same cross-validation that estimates its performance, which is optimistically biased. Here the **whole procedure** is evaluated with nested recording-grouped cross-validation: in each outer training fold, every candidate is assessed by an inner grouped CV (with its own inner threshold selection), the selection rule picks one configuration (CPU rule: highest predicted S on the CPU among candidates with 100% inner recall; GPU rule: the same with the GPU latency), and only the picked configuration is evaluated on the outer test fold. The outer test folds are never used for selection.

In [ ]:
CANDIDATES = list(itertools.product(ALL_CONFIGS, CONFIG["STRATEGIES"]))
def nested_seed(seed):
    def _run():
        outer = splitter("group", CONFIG["N_FOLDS"], seed)
        dec = {d_: np.zeros(len(y), bool) for d_ in DEVICES}; cost = {d_: np.zeros(len(y)) for d_ in DEVICES}
        sel_rows = []
        for k, (tr, te) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
            C = []
            for (b, s), st_ in CANDIDATES:
                R = run_cv_full(b, s, st_, seed + 7, idx=tr, n_outer=CONFIG["NESTED_INNER_FOLDS"], r_targets=[1.0])
                m = R["fold"] >= 0; d_in = R["oof"][m] >= R["taus"][1.0][m]; yy = y[m]
                q = d_in.mean()
                C.append({"gate": b, "scale": s, "strategy": st_, "inner_recall": d_in[yy == 1].mean(), "inner_q": q,
                          **{f"inner_S_{dv}": 1 / (rho_of(b, s, dv) + q) for dv in DEVICES}})
            C = pd.DataFrame(C)
            for dv in DEVICES:
                ok = C[C.inner_recall >= 0.9999]
                pick = (ok.sort_values(f"inner_S_{dv}", ascending=False) if len(ok)
                        else C.sort_values(["inner_recall", f"inner_S_{dv}"], ascending=False)).iloc[0]
                b, s, st_ = pick.gate, float(pick.scale), pick.strategy
                F, Ff = FEATS[(b, s)]
                tau = inner_taus(tr, F, Ff, st_, seed)[1.0]
                h = Head(st_, seed).fit(F[tr], Ff[tr], y[tr])
                dd = h.score(F[te]) >= tau
                dec[dv][te] = dd; cost[dv][te] = rho_of(b, s, dv) + dd
                yt = y[te]
                sel_rows.append({"seed": seed, "fold": k, "rule": dv, "gate": b, "scale": s, "strategy": st_,
                                 "inner_recall": pick.inner_recall, "inner_S": pick[f"inner_S_{dv}"],
                                 "outer_recall": dd[yt == 1].mean(), "outer_fpr": dd[yt == 0].mean(), "outer_q": dd.mean(),
                                 "outer_missed": int((~dd & (yt == 1)).sum())})
            print(f"nested seed {seed} fold {k}: done")
        return {"sel": pd.DataFrame(sel_rows), "dec": dec, "cost": cost}
    return cached(f"nested_seed{seed}", _run)

NESTED = {sd: nested_seed(sd) for sd in CONFIG["NESTED_SEEDS"]}
nest_rows = []
for sd, out in NESTED.items():
    for dv in DEVICES:
        dd = out["dec"][dv]
        nest_rows.append({"seed": sd, "rule": dv, "recall": dd[y == 1].mean(), "missed": int((~dd & (y == 1)).sum()),
                          "fpr": dd[y == 0].mean(), "q": dd.mean(), "S": 1 / out["cost"][dv].mean()})
nest_df = pd.DataFrame(nest_rows)
nest_sum = nest_df.groupby("rule").agg(["mean", "std"]).drop(columns="seed", level=0)
display(nest_sum.round(3))
nest_df.round(4).to_csv(os.path.join(DIRS["tables"], "nested_selection_per_seed.csv"), index=False)
sel_all = pd.concat([o["sel"] for o in NESTED.values()])
sel_all.round(4).to_csv(os.path.join(DIRS["tables"], "nested_selection_folds.csv"), index=False)
freq = sel_all.groupby(["rule", "gate", "scale", "strategy"]).size().rename("times_selected").reset_index()
save_table(freq.sort_values(["rule", "times_selected"], ascending=[True, False]), "nested_selection_frequency")
apparent = {dv: (cv_mean.loc[BEST if dv == "CPU" else (GBEST or BEST)][["recall_r100", "fpr_r100", "q_r100"]].tolist()
                 + [cv_mean.loc[BEST if dv == "CPU" else (GBEST or BEST)][f"S_{dv}_r100"]]) for dv in DEVICES}
nest_tbl = pd.DataFrame([{"rule": dv, "estimate": est, "recall": v[0], "fpr": v[1], "q": v[2], "S": v[3]}
                         for dv in DEVICES for est, v in [("apparent (selected on the same CV)", apparent[dv]),
                                                          ("nested (unbiased)", [nest_sum.loc[dv, (m_, "mean")] for m_ in ["recall", "fpr", "q", "S"]])]])
save_table(nest_tbl.round(4), "nested_vs_apparent")
record("nested", {"summary": nest_df.to_dict(orient="records"), "frequency": freq.to_dict(orient="records")})

## 17. Threshold stability and alternative threshold rules (Reviewer 1.1, Reviewer 2.5, Reviewer 3)
* **Stability:** distribution of the selected tau over the 15 outer folds; and a leave-one-joint-recording-out jackknife inside each training fold (seed 0), which shows how much tau moves when one positive recording is removed, and how many test joints the most extreme jackknife threshold would have missed.
* **Rules compared** (same folds, same heads): inner-CV threshold with margin (ours); inner-CV threshold without margin (the lowest inner joint score); **split-conformal** threshold (head fitted on two thirds of the training recordings, tau = lowest joint score among the held-out third; under exchangeability a new joint scores >= tau with probability >= n_cal/(n_cal+1)); inner-CV threshold for a 95% recall target; fixed 0.5.

In [ ]:
tau_stats = (pf_df[pf_df.apply(lambda r: (r.gate, r.scale, r.strategy) in set(SEL.values()), axis=1)]
             .assign(config=lambda d: d.gate + "@" + d.scale.astype(str) + " " + d.strategy)
             .groupby("config").tau.agg(["mean", "std", "min", "max"]).reset_index())
save_table(tau_stats.round(4), "threshold_distribution")

def jackknife(c, seed=0):
    def _run():
        b, s, st_ = c; F, Ff = FEATS[(b, s)]; rows = []
        for k, (tr, te) in enumerate(splitter("group", CONFIG["N_FOLDS"], seed).split(np.zeros(len(y)), y, groups)):
            tau0 = inner_taus(tr, F, Ff, st_, seed)[1.0]
            sc_te = Head(st_, seed).fit(F[tr], Ff[tr], y[tr]).score(F[te])
            for r in np.unique(groups[tr][y[tr] == 1]):
                trj = tr[groups[tr] != r]
                tj = inner_taus(trj, F, Ff, st_, seed)[1.0]
                rows.append({"fold": k, "left_out": r, "tau_full": tau0, "tau_jack": tj,
                             "missed_full": int(((sc_te < tau0) & (y[te] == 1)).sum()),
                             "missed_jack": int(((sc_te < tj) & (y[te] == 1)).sum()),
                             "fp_full": int(((sc_te >= tau0) & (y[te] == 0)).sum()), "fp_jack": int(((sc_te >= tj) & (y[te] == 0)).sum())})
        return pd.DataFrame(rows)
    return cached(f"jackknife_{b_s_key(c)}", _run)
def b_s_key(c):
    return f"{c[0]}_{c[1]}_{c[2]}"
jk = {tag: jackknife(c) for tag, c in SEL.items() if tag in ("best", "gpu_best")}
jk_rows = []
for tag, d in jk.items():
    rel = (d.tau_jack - d.tau_full).abs()
    worst = d.groupby("fold").missed_jack.max().sum()
    jk_rows.append({"config": cfg_str(SEL[tag]), "role": tag, "n_jackknife": len(d), "median_abs_tau_change": rel.median(),
                    "max_abs_tau_change": rel.max(), "missed_with_full_tau": int(d.groupby("fold").missed_full.first().sum()),
                    "missed_worst_jackknife_tau": int(worst), "share_jack_tau_missing_any": float((d.missed_jack > d.missed_full).mean())})
jk_df = pd.DataFrame(jk_rows); save_table(jk_df.round(4), "threshold_jackknife")

RULES = ["inner-CV + margin (ours)", "inner-CV, no margin", "split-conformal", "inner-CV, r*=95%", "fixed 0.5"]
def rules_eval(c, seed):
    b, s, st_ = c; F, Ff = FEATS[(b, s)]
    dec = {r: np.zeros(len(y), bool) for r in RULES}; ncal = []
    for tr, te in splitter("group", CONFIG["N_FOLDS"], seed).split(np.zeros(len(y)), y, groups):
        t_ = inner_taus(tr, F, Ff, st_, seed, r_targets=(1.0, 0.95))
        sc = Head(st_, seed).fit(F[tr], Ff[tr], y[tr]).score(F[te])
        dec["inner-CV + margin (ours)"][te] = sc >= t_[1.0]
        dec["inner-CV, no margin"][te] = sc >= t_["min_pos"]
        dec["inner-CV, r*=95%"][te] = sc >= t_[0.95]
        dec["fixed 0.5"][te] = sc >= 0.5
        fit_i, cal_i = next(splitter("group", 3, seed + 200).split(tr, y[tr], groups[tr]))
        hc = Head(st_, seed).fit(F[tr[fit_i]], Ff[tr[fit_i]], y[tr[fit_i]])
        cs = hc.score(F[tr[cal_i]]); pos = cs[y[tr[cal_i]] == 1]
        ncal.append(len(pos))
        dec["split-conformal"][te] = hc.score(F[te]) >= (pos.min() if len(pos) else np.inf)
    return dec, ncal
def _rules():
    rows = []
    for tag, c in SEL.items():
        if c[2].startswith("OneClass"):
            continue
        for seed in CONFIG["CV_SEEDS"]:
            dec, ncal = rules_eval(c, seed)
            for r, dd in dec.items():
                q = dd.mean()
                rows.append({"config": cfg_str(c), "role": tag, "seed": seed, "rule": r, "recall": dd[y == 1].mean(),
                             "missed": int((~dd & (y == 1)).sum()), "fpr": dd[y == 0].mean(), "q": q,
                             **{f"S_{dv}": 1 / (rho_of(c[0], c[1], dv) + q) for dv in DEVICES},
                             "conformal_bound": (np.mean(ncal) / (np.mean(ncal) + 1)) if r == "split-conformal" else np.nan})
    return pd.DataFrame(rows)
rules_df = cached(f"threshold_rules_{SIGN}", _rules)
rules_sum = rules_df.groupby(["config", "role", "rule"]).agg({"recall": ["mean", "min"], "missed": ["mean", "max"], "fpr": "mean", "q": "mean",
                                                             "conformal_bound": "mean"})
rules_sum.columns = ["_".join(c_) for c_ in rules_sum.columns]
rules_sum = rules_sum.reset_index()
for dv in DEVICES:
    rules_sum[f"S_{dv}"] = [1 / (rho_of(*SEL[r.role][:2], dv) + r.q_mean) for r in rules_sum.itertuples()]
save_table(rules_sum.round(4), "threshold_rules")

## 18. Unseen stations: few-shot calibration with separate calibration and test recordings (Reviewer 2.3, Reviewer 3)
For each held-out station, the gate is trained on the other stations. The station's recordings are split at random into **calibration** and **test** recordings (`CALIB_SPLITS` random splits); results are measured on the test recordings only. Variants: (A) source threshold only (no target data); (B) threshold lowered to the lowest calibration-joint score of the target station when that is lower; (C) head re-trained with the calibration frames added, threshold from inner CV on the combined data.

In [ ]:
def station_calibration(c, seed=0):
    def _run():
        b, s, st_ = c; F, Ff = FEATS[(b, s)]; rng = np.random.default_rng(seed); rows = []
        for hs in sorted(np.unique(stations)):
            tgt = np.where(stations == hs)[0]; src = np.where(stations != hs)[0]
            recs = np.unique(groups[tgt])
            if y[tgt].sum() == 0 or len(recs) < 2:
                continue
            tau_src = inner_taus(src, F, Ff, st_, seed)[1.0]
            h0 = Head(st_, seed).fit(F[src], Ff[src], y[src])
            sc0 = h0.score(F)
            for rep_ in range(CONFIG["CALIB_SPLITS"]):
                cal_r = rng.permutation(recs)[:len(recs) // 2]
                cal = tgt[np.isin(groups[tgt], cal_r)]; test = tgt[~np.isin(groups[tgt], cal_r)]
                if y[test].sum() == 0:
                    continue
                pos_cal = sc0[cal][y[cal] == 1]
                tau_b = min(tau_src, pos_cal.min()) if len(pos_cal) else tau_src
                comb = np.r_[src, cal]
                tau_c = inner_taus(comb, F, Ff, st_, seed)[1.0]
                sc_c = Head(st_, seed).fit(F[comb], Ff[comb], y[comb]).score(F[test])
                for var, sc_t, tau in [("A: source threshold", sc0[test], tau_src), ("B: threshold calibrated on station", sc0[test], tau_b),
                                       ("C: head + threshold updated", sc_c, tau_c)]:
                    d = sc_t >= tau; yt = y[test]
                    rows.append({"station": hs, "split": rep_, "variant": var, "test_joints": int(yt.sum()), "test_background": int((yt == 0).sum()),
                                 "calib_joints": int(y[cal].sum()), "detected": int((d & (yt == 1)).sum()),
                                 "false_pos": int((d & (yt == 0)).sum())})
        return pd.DataFrame(rows)
    return cached(f"station_calib_{b_s_key(c)}", _run)
calib_rows = []
for tag, c in SEL.items():
    if c[2].startswith("OneClass"):
        continue
    d = station_calibration(c)
    if not len(d):
        continue
    g = d.groupby(["station", "variant"]).agg(test_joints=("test_joints", "sum"), detected=("detected", "sum"),
                                               test_background=("test_background", "sum"), false_pos=("false_pos", "sum")).reset_index()
    g["recall"] = g.detected / g.test_joints; g["fpr"] = np.where(g.test_background > 0, g.false_pos / g.test_background.clip(lower=1), np.nan)
    tot = d.groupby("variant").agg(test_joints=("test_joints", "sum"), detected=("detected", "sum"),
                                   test_background=("test_background", "sum"), false_pos=("false_pos", "sum")).reset_index()
    tot["station"] = "all"; tot["recall"] = tot.detected / tot.test_joints
    tot["fpr"] = tot.false_pos / tot.test_background.clip(lower=1)
    calib_rows.append(pd.concat([g, tot]).assign(config=cfg_str(c), role=tag))
calib_df = pd.concat(calib_rows) if calib_rows else pd.DataFrame()
save_table(calib_df.round(4), "station_calibration")

## 19. Robustness to operating conditions and augmentation (Reviewer 2.9, Reviewer 2.3)
Test frames are perturbed (illumination, contrast, simulated arc glare, sensor noise, motion and defocus blur, small rotation, scale/shift, JPEG compression); thresholds are selected on clean frames, as in deployment. Each gate is trained either on clean frames (+ mirror) or with `AUG_COPIES` random photometric/geometric copies of every training frame added (parameters drawn from ranges that differ from the test perturbations). The augmented heads are also evaluated leave-one-station-out.

In [ ]:
import zlib
def _u8(img):
    return np.clip(img, 0, 255).astype(np.uint8)
def _glare(img, rng, n=2, amp=220.0):
    h, w = img.shape[:2]; yy_, xx_ = np.mgrid[0:h, 0:w]; out = img.astype(np.float32).copy()
    for _ in range(n):
        cy, cx, sg = rng.uniform(0, h), rng.uniform(0, w), rng.uniform(10, 28)
        out += amp * np.exp(-((yy_ - cy) ** 2 + (xx_ - cx) ** 2) / (2 * sg ** 2))[..., None]
    return np.clip(out, 0, 255)
def _affine(img, ang=0.0, sc=1.0, tx=0.0, ty=0.0):
    h, w = img.shape[:2]; M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc); M[:, 2] += (tx, ty)
    return cv2.warpAffine(img.astype(np.float32), M, (w, h), borderMode=cv2.BORDER_REFLECT)
def perturb(img, kind, rng):
    f = img.astype(np.float32)
    if kind == "brightness-30": return np.clip(f * 0.7, 0, 255)
    if kind == "brightness+30": return np.clip(f * 1.3, 0, 255)
    if kind.startswith("gamma"): return 255 * (f / 255) ** float(kind[5:])
    if kind == "contrast0.6": return np.clip((f - f.mean()) * 0.6 + f.mean(), 0, 255)
    if kind.startswith("noise"): return np.clip(f + rng.normal(0, float(kind[5:]), f.shape), 0, 255)
    if kind == "glare": return _glare(f, rng)
    if kind == "motion_blur": k = np.zeros((9, 9), np.float32); k[4, :] = 1 / 9; return cv2.filter2D(f, -1, k)
    if kind == "defocus": return cv2.GaussianBlur(f, (7, 7), 2.0)
    if kind == "rotate5": return _affine(f, ang=5.0)
    if kind == "scale_shift": return _affine(f, sc=1.1, tx=8, ty=8)
    if kind == "jpeg30":
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(_u8(f), cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 30])
        return cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB).astype(np.float32)
    raise ValueError(kind)
def random_aug(img, rng):
    f = img.astype(np.float32)
    f = np.clip(f * rng.uniform(0.75, 1.25), 0, 255)
    f = 255 * (f / 255) ** rng.uniform(0.75, 1.35)
    if rng.random() < 0.5: f = np.clip(f + rng.normal(0, rng.uniform(2, 12), f.shape), 0, 255)
    if rng.random() < 0.4: f = _glare(f, rng, n=1, amp=rng.uniform(80, 200))
    if rng.random() < 0.3: f = cv2.GaussianBlur(f, (5, 5), rng.uniform(0.5, 1.5))
    return _affine(f, ang=rng.uniform(-4, 4), sc=rng.uniform(0.95, 1.05), tx=rng.uniform(-6, 6), ty=rng.uniform(-6, 6))

ROB_CFGS = list(dict.fromkeys(c for c in SEL.values() if not c[2].startswith("OneClass")))
ROB_BACKBONES = list(dict.fromkeys((c[0], c[1]) for c in ROB_CFGS))
def _feats_variants():
    PF, AF = {}, {}
    for b, s in ROB_BACKBONES:
        for p_ in CONFIG["PERTURBATIONS"]:
            rng = np.random.default_rng(zlib.crc32(p_.encode()))
            PF[(b, s, p_)] = extract(b, s, np.stack([perturb(im, p_, rng) for im in X]))
        for c_ in range(CONFIG["AUG_COPIES"]):
            rng = np.random.default_rng(1000 + c_)
            AF[(b, s, c_)] = extract(b, s, np.stack([random_aug(im, rng) for im in X]))
        print(f"perturbed/augmented features: {b}@{s} done")
    return PF, AF
PFEATS, AFEATS = cached("perturbation_features_" + hashlib.md5(repr(ROB_BACKBONES).encode()).hexdigest()[:6], _feats_variants)

def _robust():
    rows, logo = [], []
    for c in ROB_CFGS:
        b, s, st_ = c
        tests_ = {p_: PFEATS[(b, s, p_)] for p_ in CONFIG["PERTURBATIONS"]}
        aug = [AFEATS[(b, s, k)] for k in range(CONFIG["AUG_COPIES"])]
        for train in ["clean", "augmented"]:
            ex = None if train == "clean" else aug
            for seed in CONFIG["CV_SEEDS"]:
                R = run_cv_full(b, s, st_, seed, test_sets=tests_, extra=ex, r_targets=[1.0])
                t_ = R["taus"][1.0]
                for p_, sc in [("none", R["oof"]), ("mirror", R["flip"])] + list(R["tests"].items()):
                    dd = sc >= t_
                    rows.append({"config": cfg_str(c), "training": train, "perturbation": p_, "seed": seed,
                                 "recall": dd[y == 1].mean(), "fpr": dd[y == 0].mean(), "q": dd.mean()})
            R = run_cv_full(b, s, st_, CONFIG["CV_SEEDS"][0], outer=LeaveOneGroupOut(), extra=ex, r_targets=[1.0])
            dd = R["oof"] >= R["taus"][1.0]
            logo.append({"config": cfg_str(c), "training": train, "recall": dd[y == 1].mean(), "missed": int((~dd & (y == 1)).sum()),
                         "fpr": dd[y == 0].mean(), "q": dd.mean()})
    return pd.DataFrame(rows), pd.DataFrame(logo)
pert_raw, logo_aug = cached(f"robustness_perturbations_{SIGN}", _robust)
pert_df = pert_raw.groupby(["config", "training", "perturbation"], sort=False)[["recall", "fpr", "q"]].mean().reset_index()
save_table(pert_df.round(4), "perturbation_robustness")
save_table(logo_aug.round(4), "station_transfer_augmentation")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=False)
c0 = cfg_str(ROB_CFGS[0])
for ax, met in zip(axes, ["recall", "fpr"]):
    d = pert_df[pert_df.config == c0].pivot(index="perturbation", columns="training", values=met)
    order = ["none", "mirror"] + CONFIG["PERTURBATIONS"]; d = d.reindex([o for o in order if o in d.index])
    d.plot.bar(ax=ax, color=["#9ecae1", "#e6550d"], width=0.75)
    ax.set_ylabel({"recall": "Recall (target 100%)", "fpr": "False-positive rate"}[met]); ax.set_xlabel("")
    ax.legend(frameon=False, fontsize=7, title=None)
axes[0].set_title(c0, fontsize=8)
fig.tight_layout(); save_fig(fig, "fig_perturbation_robustness"); plt.show()

## 20. Baseline gates (Reviewer 2.4)
* **Random or periodic sampling** (a frame filter that ignores content): routing a share q of frames finds on average a share q of the joints, so full recall needs q = 1 and S = 1.
* **Handcrafted gates** (HOG; intensity/edge/gradient statistics) are part of the design space from Section 8 on and appear in the tables with every head.
* **Fine-tuned CNN gate:** MobileNetV3-Small trained end to end (all layers, weighted binary cross-entropy, Adam), with the same grouped CV and inner threshold selection (`FT_SEEDS`). Its cost equals the frozen MobileNetV3-Small gate.
Object detectors are not evaluated because the frames have no bounding-box labels.

In [ ]:
import gc
def _predict(m, idx, bs=32):
    # plain batched forward pass (avoids Keras predict() re-tracing and its memory growth)
    return np.concatenate([m(tf.constant(X[idx[k:k + bs]]), training=False).numpy().ravel() for k in range(0, len(idx), bs)])
def finetune_scores(tr, te_list, seed):
    keras.backend.clear_session(); gc.collect()
    keras.utils.set_random_seed(seed)
    shp = input_shape(1.0)
    base = tf.keras.applications.MobileNetV3Small(input_shape=shp, include_top=False, weights=CONFIG["WEIGHTS"],
                                                  pooling="avg", include_preprocessing=True)
    base.trainable = True
    inp = keras.Input(shape=shp)
    x = keras.layers.RandomFlip("horizontal", seed=seed)(inp)          # mirror augmentation on the fly (no copied arrays)
    out = keras.layers.Dense(1, activation="sigmoid")(base(x, training=False))
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(CONFIG["FT_LR"]), loss="binary_crossentropy")
    ytr = y[tr].astype(np.float32)
    cw = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * max((ytr == 1).sum(), 1))}
    ds = tf.data.Dataset.from_tensor_slices((tr, ytr)).shuffle(len(tr), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda i, t: (tf.gather(X_TF, i), t)).batch(CONFIG["FT_BATCH"]).prefetch(2)
    m.fit(ds, epochs=CONFIG["FT_EPOCHS"], class_weight=cw, verbose=0)
    outs = [_predict(m, np.asarray(t_)) for t_ in te_list]
    del m, base, ds; keras.backend.clear_session(); gc.collect()
    return outs
X_TF = tf.constant(X) if CONFIG["FINETUNE"] else None
def _finetune_fold(seed, k, tr, te):
    def _run():
        inner_sc = np.full(len(tr), np.nan)
        for itr, ite in splitter("group", CONFIG["INNER_FOLDS"], seed + 100).split(tr, y[tr], groups[tr]):
            inner_sc[ite] = finetune_scores(tr[itr], [tr[ite]], seed)[0]
        ok = ~np.isnan(inner_sc); tau = select_tau(inner_sc[ok], y[tr][ok], 1.0)
        return {"te": te, "score": finetune_scores(tr, [te], seed)[0], "tau": tau}
    return cached(f"finetune_seed{seed}_fold{k}", _run)          # one checkpoint per fold: a crash loses at most one fold
def _finetune():
    rows, pf = [], []
    for seed in CONFIG["FT_SEEDS"]:
        oof = np.full(len(y), np.nan); dec = np.zeros(len(y), bool)
        for k, (tr, te) in enumerate(splitter("group", CONFIG["N_FOLDS"], seed).split(np.zeros(len(y)), y, groups)):
            r_ = _finetune_fold(seed, k, tr, te)
            oof[te] = r_["score"]; dec[te] = oof[te] >= r_["tau"]
            pf.append({"seed": seed, "fold": k, "tau": r_["tau"], "pr_auc": average_precision_score(y[te], oof[te])})
            print(f"fine-tuning seed {seed} fold {k}: done")
        q = dec.mean()
        rows.append({"seed": seed, "pr_auc": average_precision_score(y, oof), "fpr_at_r100": fpr_at_recall(oof, y, 1.0),
                     "recall": dec[y == 1].mean(), "missed": int((~dec & (y == 1)).sum()), "fpr": dec[y == 0].mean(), "q": q})
    return pd.DataFrame(rows), pd.DataFrame(pf)
if CONFIG["FINETUNE"]:
    ft_df, ft_pf = cached("finetune", _finetune)
    save_table(ft_df.round(4), "finetuned_gate")
else:
    ft_df = None

base_rows = [{"gate": "Random / periodic sampling", "head": "-", "recall": 1.0, "fpr": 1.0, "q": 1.0,
              **{f"S_{dv}": 1.0 for dv in DEVICES}, "note": "full recall requires routing every frame"}]
for name in HANDCRAFTED:
    sub = cv_mean.loc[name] if name in cv_mean.index.get_level_values(0) else None
    if sub is None:
        continue
    sup_ = sub[[not i[1].startswith("OneClass") for i in sub.index]] if isinstance(sub.index, pd.MultiIndex) else sub
    r_ = sup_.sort_values(["recall_r100", "S_CPU_r100"], ascending=False).iloc[0]
    base_rows.append({"gate": name, "head": r_.name[-1] if isinstance(r_.name, tuple) else r_.name, "pr_auc": r_.pr_auc,
                      "recall": r_.recall_r100, "fpr": r_.fpr_r100, "q": r_.q_r100, **{f"S_{dv}": r_[f"S_{dv}_r100"] for dv in DEVICES}})
if ft_df is not None:
    q_ = ft_df.q.mean()
    base_rows.append({"gate": "MobileNetV3Small fine-tuned", "head": "end-to-end", "pr_auc": ft_df.pr_auc.mean(),
                      "recall": ft_df.recall.mean(), "fpr": ft_df.fpr.mean(), "q": q_,
                      **{f"S_{dv}": 1 / (rho_of("MobileNetV3Small", 1.0, dv) + q_) for dv in DEVICES}})
for tag, c in SEL.items():
    m_ = cv_mean.loc[c]
    base_rows.append({"gate": f"{c[0]}@{c[1]} (frozen)", "head": c[2], "pr_auc": m_.pr_auc, "recall": m_.recall_r100,
                      "fpr": m_.fpr_r100, "q": m_.q_r100, **{f"S_{dv}": m_[f"S_{dv}_r100"] for dv in DEVICES}, "note": tag})
base_df = pd.DataFrame(base_rows); save_table(base_df.round(4), "baselines")

## 21. HED on the GPU: a PyTorch port of the Caffe model (Reviewer 1.5, Reviewer 3)
The pip build of OpenCV has no CUDA backend, so HED ran on the CPU in v3. Here the Caffe graph is rebuilt in PyTorch from `deploy.prototxt`, with the weights read from the Caffe model through OpenCV, and run on the GPU. The port is accepted only if its output matches OpenCV's CPU output (max absolute difference < 1e-3 on the edge probabilities); otherwise the GPU-HED sections are skipped.

In [ ]:
import torch
# v4.1: weights are read directly from the .caffemodel file (OpenCV >= 4.13 no longer exposes layers by name)
def _varint(b, i):
    r = s = 0
    while True:
        c = b[i]; i += 1; r |= (c & 0x7F) << s; s += 7
        if c < 0x80:
            return r, i
def _fields(b, start=0, end=None):
    i, end = start, (len(b) if end is None else end)
    while i < end:
        key, i = _varint(b, i); f, wt = key >> 3, key & 7
        if wt == 0:
            v, i = _varint(b, i); yield f, wt, v
        elif wt == 1:
            yield f, wt, (i, i + 8); i += 8
        elif wt == 2:
            L, i = _varint(b, i); yield f, wt, (i, i + L); i += L
        elif wt == 5:
            yield f, wt, (i, i + 4); i += 4
        else:
            raise ValueError(f"unsupported wire type {wt}")
def _blob(b, s, e):
    dims, legacy, data, i = [], {}, [], s
    u8 = None
    while i < e:
        key, j = _varint(b, i); f, wt = key >> 3, key & 7
        if f == 5 and wt == 5:                       # unpacked repeated float: run of [0x2D, 4 bytes]
            if u8 is None:
                u8 = np.frombuffer(b, np.uint8)
            n = (e - i) // 5
            tags = u8[i:i + 5 * n:5]
            run = n if np.all(tags == 0x2D) else int(np.argmin(tags == 0x2D))
            data.append(np.frombuffer(b[i:i + 5 * run], np.uint8).reshape(run, 5)[:, 1:].copy().view("<f4").ravel())
            i += 5 * run; continue
        i = j
        if wt == 0:
            v, i = _varint(b, i)
            if f in (1, 2, 3, 4): legacy[f] = v
        elif wt == 2:
            L, i = _varint(b, i)
            if f == 5: data.append(np.frombuffer(b[i:i + L], "<f4"))
            elif f == 8: data.append(np.frombuffer(b[i:i + L], "<f8").astype(np.float32))
            elif f == 7:
                for f2, wt2, v2 in _fields(b, i, i + L):
                    if f2 == 1 and wt2 == 2:
                        k = v2[0]
                        while k < v2[1]:
                            d, k = _varint(b, k); dims.append(d)
                    elif f2 == 1 and wt2 == 0:
                        dims.append(v2)
            i += L
        elif wt == 5: i += 4
        elif wt == 1: i += 8
    arr = np.concatenate(data).astype(np.float32) if data else np.zeros(0, np.float32)
    if not dims:
        dims = [legacy.get(k, 1) for k in (1, 2, 3, 4)]
    return arr.reshape(dims) if int(np.prod(dims)) == arr.size else arr
def read_caffemodel(path):
    # minimal reader of a Caffe .caffemodel (NetParameter protobuf): {layer name: [blob arrays]}
    b = open(path, "rb").read(); out = {}
    for f, wt, v in _fields(b):
        if wt != 2 or f not in (2, 100):              # 2: V1LayerParameter, 100: LayerParameter
            continue
        name, blobs = None, []
        for f2, wt2, v2 in _fields(b, *v):
            if wt2 == 2 and ((f == 100 and f2 == 1) or (f == 2 and f2 == 4)):
                name = b[v2[0]:v2[1]].decode()
            elif wt2 == 2 and ((f == 100 and f2 == 7) or (f == 2 and f2 == 6)):
                blobs.append(_blob(b, *v2))
        if name and blobs:
            out[name] = blobs
    return out

def parse_prototxt(path):
    txt = re.sub(r"#.*", "", open(path).read())
    toks = re.findall(r'"[^"]*"|\'[^\']*\'|[{}:]|[^\s{}:"\']+', txt)      # names may be in double or single quotes
    pos = 0
    def block():
        nonlocal pos
        out = []
        while pos < len(toks) and toks[pos] != "}":
            key = toks[pos]; pos += 1
            if toks[pos] == ":":
                pos += 1
            if toks[pos] == "{":
                pos += 1; val = block(); pos += 1
            else:
                val = toks[pos].strip('"').strip("'"); pos += 1
            out.append((key, val))
        return out
    return block()
def _get(kv, key, default=None):
    v = [x for k, x in kv if k == key]
    return v[0] if v else default
def _all(kv, key):
    return [x for k, x in kv if k == key]
CAFFE_ENUM = {"CONVOLUTION": "Convolution", "DECONVOLUTION": "Deconvolution", "RELU": "ReLU", "POOLING": "Pooling",
              "CROP": "Crop", "CONCAT": "Concat", "SIGMOID": "Sigmoid", "DROPOUT": "Dropout", "INPUT": "Input"}

class CaffeTorch(torch.nn.Module):
    def __init__(self, prototxt, weights):
        super().__init__()
        net = parse_prototxt(prototxt)
        self.layers = _all(net, "layer") + _all(net, "layers")
        self.params = torch.nn.ParameterDict(); self.spec = []
        for L in self.layers:
            name, typ = _get(L, "name"), _get(L, "type"); typ = CAFFE_ENUM.get(typ, typ)
            spec = {"name": name, "type": typ, "bottom": _all(L, "bottom"), "top": _all(L, "top")}
            if typ in ("Convolution", "Deconvolution"):
                cp = _get(L, "convolution_param", [])
                spec.update(k=int(_get(cp, "kernel_size", 1)), s=int(_get(cp, "stride", 1)), p=int(_get(cp, "pad", 0)),
                            g=int(_get(cp, "group", 1)), d=int(_get(cp, "dilation", 1)))
                if name not in weights:
                    raise KeyError(f"layer {name} not in the caffemodel (has: {sorted(weights)[:8]}...)")
                w = weights[name][0]
                bvec = weights[name][1].ravel() if len(weights[name]) > 1 else None
                key = name.replace("-", "_").replace(".", "_")
                self.params[key + "_w"] = torch.nn.Parameter(torch.from_numpy(np.ascontiguousarray(w, np.float32)), requires_grad=False)
                if bvec is not None and bvec.size:
                    self.params[key + "_b"] = torch.nn.Parameter(torch.from_numpy(np.ascontiguousarray(bvec, np.float32)), requires_grad=False)
                spec["key"] = key
            elif typ == "Pooling":
                pp = _get(L, "pooling_param", [])
                spec.update(pool=_get(pp, "pool", "MAX"), k=int(_get(pp, "kernel_size", 2)), s=int(_get(pp, "stride", 1)), p=int(_get(pp, "pad", 0)))
            self.spec.append(spec)
        self.out_name = self.spec[-1]["top"][0]
        self.in_name = _get(net, "input", "data")
    def forward(self, x):
        B = {self.in_name: x}
        for L in self.spec:
            t = L["type"]; bt = [B[b] for b in L["bottom"]]
            if t in ("Input", "Dropout"):
                continue
            if t == "Convolution":
                o = torch.nn.functional.conv2d(bt[0], self.params[L["key"] + "_w"], self.params.get(L["key"] + "_b"),
                                               stride=L["s"], padding=L["p"], dilation=L["d"], groups=L["g"])
            elif t == "Deconvolution":
                o = torch.nn.functional.conv_transpose2d(bt[0], self.params[L["key"] + "_w"], self.params.get(L["key"] + "_b"),
                                                         stride=L["s"], padding=L["p"], groups=L["g"], dilation=L["d"])
            elif t == "ReLU":
                o = torch.relu(bt[0])
            elif t == "Pooling":
                o = (torch.nn.functional.max_pool2d if L["pool"] == "MAX" else torch.nn.functional.avg_pool2d)(
                    bt[0], L["k"], L["s"], L["p"], ceil_mode=True)
            elif t == "Crop":            # same centre crop as the CropLayer registered with OpenCV
                a, r = bt[0], bt[1]; ys = int((a.shape[2] - r.shape[2]) / 2); xs = int((a.shape[3] - r.shape[3]) / 2)
                o = a[:, :, ys:ys + r.shape[2], xs:xs + r.shape[3]]
            elif t == "Concat":
                o = torch.cat(bt, 1)
            elif t == "Sigmoid":
                o = torch.sigmoid(bt[0])
            else:
                raise NotImplementedError(t)
            B[L["top"][0]] = o
        return B[self.out_name]

HED_GPU = None; L_HED_GPU = np.nan; HED_PORT = {"ok": False}
if torch.cuda.is_available():
    try:
        dev_t = torch.device("cuda"); torch.backends.cudnn.benchmark = True
        HED_W = read_caffemodel(CONFIG["HED_MODEL"])
        print("caffemodel layers with weights:", len(HED_W))
        hed_t = CaffeTorch(CONFIG["HED_PROTOTXT"], HED_W).to(dev_t).eval()
        mean_t = torch.tensor(CONFIG["HED_MEAN"], dtype=torch.float32, device=dev_t).view(1, 3, 1, 1)
        @torch.no_grad()
        def hed_gpu_forward(bgr):
            x = torch.from_numpy(np.ascontiguousarray(bgr.transpose(2, 0, 1)[None], np.float32)).to(dev_t) - mean_t
            o = hed_t(x); torch.cuda.synchronize()
            return o.cpu().numpy()
        diffs = []
        for i in np.linspace(0, len(X) - 1, 5).astype(int):
            bgr = cv2.cvtColor(np.clip(X[i], 0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            a, b_ = hed_forward(hed, bgr), hed_gpu_forward(bgr)
            diffs.append(float(np.abs(a.reshape(-1) - b_.reshape(-1)).max()) if a.size == b_.size else np.inf)
        HED_PORT = {"ok": max(diffs) < 1e-3, "max_abs_diff": max(diffs)}
        if HED_PORT["ok"]:
            HED_GPU = hed_gpu_forward
            t_ = time_calls(HED_GPU, bgr1, CONFIG["LAT_WARMUP"], CONFIG["HED_REPEATS"] * 3)
            L_HED_GPU = float(t_.mean())
            lat = pd.concat([lat, pd.DataFrame([{"model": "HED (PyTorch port)", "scale": 1.0, "device": "GPU", "mean_ms": 1e3 * t_.mean(),
                                                 "median_ms": 1e3 * np.median(t_), "sd_ms": 1e3 * t_.std(), "rho": t_.mean() / L_HED}])],
                            ignore_index=True)
            save_table(lat.round(4), "latency_cpu_gpu")
    except Exception as e:
        HED_PORT = {"ok": False, "error": repr(e)}
        try:
            print("prototxt layers:", [(_get(L, "name"), _get(L, "type")) for L in (_all(parse_prototxt(CONFIG["HED_PROTOTXT"]), "layer") + _all(parse_prototxt(CONFIG["HED_PROTOTXT"]), "layers"))][:40])
        except Exception as e2:
            print("prototxt could not be parsed:", repr(e2))
print("HED GPU port:", HED_PORT, "| HED latency GPU (ms):", 1e3 * L_HED_GPU)
record("hed_gpu_port", HED_PORT); record("L_HED_gpu_s", L_HED_GPU)

## 22. End-to-end validation in three deployment settings, all repetitions, stream replay (Reviewer 1.9, Reviewer 2.6, Reviewer 3)
Frames are processed **in stream order** (recording, then frame number), one at a time. Settings: **CPU** (gate and HED on the CPU); **hybrid** (gate on the GPU, HED on the CPU, as in v3); **GPU** (gate and HED on the GPU, if the port of Section 21 passed). For each selected gate and each repetition, routing uses that repetition's cross-validated decisions; the prediction S = 1/(rho + q) uses the same q, with rho and the reference HED latency of the setting. Per-frame latencies are kept to characterise streaming: sustained throughput, latency percentiles, and the queueing delay at given camera frame rates (Lindley recursion).

In [ ]:
ORDER = np.lexsort((meta.frame_no.values, meta.recording.values))
BGR = [cv2.cvtColor(np.clip(im, 0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR) for im in X]
SETTINGS = [("CPU", "CPU", "CPU")] + ([("hybrid", "GPU", "CPU")] if "GPU" in DEVICES else []) + \
           ([("GPU", "GPU", "GPU")] if HED_GPU is not None else [])
L_REF = {"CPU": L_HED, "GPU": L_HED_GPU}
def rho_setting(b, s, gate_dev, hed_dev):
    r = lat[(lat.model == b) & (lat.scale == s) & (lat.device == gate_dev)]
    return float(r.mean_ms.iloc[0]) / (1e3 * L_REF[hed_dev])
def _e2e(settings):
    rows, traces = [], {}
    order = ORDER if not SMOKE_TEST else ORDER[:12]
    for c in E2E_CFGS:
        b, s, st_ = c
        head = Head(st_).fit(FEATS[(b, s)][0], FEATS[(b, s)][1], y)
        for name, gdev, hdev in settings:
            with tf.device(f"/{gdev}:0"):
                if b in HANDCRAFTED:
                    gate_fn = lambda img, b=b: head.score(HANDCRAFTED[b](img)[None])
                else:
                    m = build_extractor(b, s); fwd = make_fwd(m)
                    gate_fn = lambda img, fwd=fwd, s=s: head.score(fwd(tf.constant(resize_batch(img[None], s))).numpy())
                hed_fn = (lambda im: hed_forward(hed, im)) if hdev == "CPU" else HED_GPU
                for _ in range(CONFIG["LAT_WARMUP"]):
                    gate_fn(X[0]); hed_fn(BGR[0])
                for seed in CONFIG["CV_SEEDS"]:
                    oof, _, taus = OOF[(*c, seed)]
                    route = oof >= taus[1.0]
                    t_all = []
                    for i in order:
                        t0 = time.perf_counter(); gate_fn(X[i])
                        if route[i]:
                            hed_fn(BGR[i])
                        t_all.append(time.perf_counter() - t0)
                    t_all = np.array(t_all); q_ = float(route[order].mean()); rho_ = rho_setting(b, s, gdev, hdev)
                    rows.append({"config": cfg_str(c), "setting": name, "gate_device": gdev, "hed_device": hdev, "seed": seed,
                                 "q": q_, "rho": rho_, "L_pipeline_s": t_all.mean(), "FPS": 1 / t_all.mean(),
                                 "S_measured": L_REF[hdev] / t_all.mean(), "S_predicted": 1 / (rho_ + q_),
                                 "p50_ms": 1e3 * np.percentile(t_all, 50), "p95_ms": 1e3 * np.percentile(t_all, 95),
                                 "p99_ms": 1e3 * np.percentile(t_all, 99), "max_ms": 1e3 * t_all.max()})
                    traces[(cfg_str(c), name, seed)] = t_all
            keras.backend.clear_session()
            print(f"end-to-end {cfg_str(c)} [{name}] done")
    return pd.DataFrame(rows), traces
E2E_CFGS = list(dict.fromkeys([c for c in (BEST, GBEST, ACC, REF) if c is not None]))
BASE_SET = [x for x in SETTINGS if x[0] != "GPU"]; GPU_SET = [x for x in SETTINGS if x[0] == "GPU"]
e2e_raw, TRACES = cached("end_to_end_" + hashlib.md5(repr((E2E_CFGS, [x[0] for x in BASE_SET])).encode()).hexdigest()[:6],
                         lambda: _e2e(BASE_SET))
if GPU_SET:
    raw_g, tr_g = cached("end_to_end_gpu_" + hashlib.md5(repr(E2E_CFGS).encode()).hexdigest()[:6], lambda: _e2e(GPU_SET))
    e2e_raw = pd.concat([e2e_raw, raw_g], ignore_index=True); TRACES = {**TRACES, **tr_g}
e2e_raw["model_error_%"] = 100 * (e2e_raw.S_predicted - e2e_raw.S_measured) / e2e_raw.S_measured
e2e_raw.round(4).to_csv(os.path.join(DIRS["tables"], "end_to_end_per_seed.csv"), index=False)
agg = {c_: "mean" for c_ in ["q", "rho", "L_pipeline_s", "FPS", "S_measured", "S_predicted", "p50_ms", "p95_ms", "p99_ms", "max_ms"]}
e2e_df = e2e_raw.groupby(["config", "setting", "gate_device", "hed_device"], sort=False).agg(agg).reset_index()
e2e_df["S_measured_sd"] = e2e_raw.groupby(["config", "setting"], sort=False).S_measured.std().values
e2e_df["model_error_%"] = 100 * (e2e_df.S_predicted - e2e_df.S_measured) / e2e_df.S_measured
e2e_df["max_abs_error_%"] = e2e_raw.groupby(["config", "setting"], sort=False)["model_error_%"].apply(lambda v: v.abs().max()).values
save_table(e2e_df.round(4), "end_to_end")
record("end_to_end", e2e_raw.to_dict(orient="records"))

# --- streaming: queueing delay at given camera frame rates (seed 0 traces) ---
q_rows = []
for (cname, setting, seed), t_all in TRACES.items():
    if seed != CONFIG["CV_SEEDS"][0]:
        continue
    for fps in CONFIG["CAMERA_FPS"]:
        dt = 1 / fps; w = 0.0; waits = []
        for t_ in t_all:
            waits.append(w); w = max(0.0, w + t_ - dt)
        waits = np.array(waits)
        q_rows.append({"config": cname, "setting": setting, "camera_fps": fps, "utilisation": t_all.mean() / dt,
                       "keeps_up": bool(t_all.mean() < dt), "p95_delay_ms": 1e3 * np.percentile(waits + t_all, 95),
                       "max_delay_ms": 1e3 * (waits + t_all).max(), "max_backlog_frames": int(np.ceil(waits.max() / dt))})
stream_df = pd.DataFrame(q_rows); save_table(stream_df.round(3), "stream_queueing")

fig, ax = plt.subplots(figsize=(6.2, 3.8))
qq = np.linspace(0.005, 1, 400)
sty = {"CPU": "#2171b5", "hybrid": "#e6550d", "GPU": "#31a354"}
LSTY = dict(zip(E2E_CFGS, ["-", "-.", ":", "--"]))
for c in E2E_CFGS:
    for name, gdev, hdev in SETTINGS:
        r_ = rho_setting(c[0], c[1], gdev, hdev)
        ax.plot(100 * qq, 1 / (r_ + qq), color=sty[name], lw=1.0, alpha=0.6, ls=LSTY[c])
for _, r in e2e_raw.iterrows():
    ax.plot(100 * r.q, r.S_measured, "o", mfc="white", mec=sty[r.setting], mew=1.4, ms=5)
from matplotlib.lines import Line2D
hh = [Line2D([], [], color=sty[n], label=f"{n} setting") for n, _, _ in SETTINGS] + \
     [Line2D([], [], color="#555", ls=LSTY[c], label=cfg_str(c)) for c in E2E_CFGS] + \
     [Line2D([], [], color="#555", marker="o", mfc="white", ls="", label="measured (each repetition)")]
ax.plot(100 * qq, 1 / qq, color="#999999", lw=0.8, ls=":")
ax.axhline(1, color="#555555", lw=0.8, ls="--"); ax.axvline(100 * PI, color="#555555", lw=0.6, ls=":")
ax.set_yscale("log"); ax.set_xlim(0, 60); ax.set_ylim(0.5, 30)
ax.set_xlabel("Frames routed to HED, q (%)"); ax.set_ylabel("Speed-up over un-gated HED (log)")
ax.legend(handles=hh, frameon=False, fontsize=6.5); fig.tight_layout(); save_fig(fig, "fig_speedup_model_validation"); plt.show()

## 23. Edge-map fidelity: gating versus cheaper edge detectors (Reviewer 2.8, Reviewer 3)
No edge annotations exist, so edge maps are compared with **full-resolution HED as the reference** on the joint frames. Gating does not change HED's output on routed frames (checked numerically), so its fidelity is 1 on every routed joint and 0 on a missed one. The alternatives make *every* frame cheaper instead: HED at reduced input resolution, Canny and Sobel. Agreement is the boundary F-score with a tolerance of `EDGE_TOL_PX` pixels (reference = HED >= 0.5, thinned), using the best global threshold for each method (ODS-style). This measures agreement with HED, not accuracy against human labels.

In [ ]:
from skimage.morphology import thin
JOINTS = np.where(y == 1)[0] if not SMOKE_TEST else np.where(y == 1)[0][:3]
def hed_map(bgr, scale=1.0):
    if scale == 1.0:
        return hed_forward(hed, bgr)[0, 0]
    small = cv2.resize(bgr, (int(round(bgr.shape[1] * scale)), int(round(bgr.shape[0] * scale))), interpolation=cv2.INTER_AREA)
    return cv2.resize(hed_forward(hed, small)[0, 0], (bgr.shape[1], bgr.shape[0]), interpolation=cv2.INTER_LINEAR)
def bf_counts(pred, ref, tol):
    dt_ref = cv2.distanceTransform((~ref).astype(np.uint8), cv2.DIST_L2, 3)
    dt_pred = cv2.distanceTransform((~pred).astype(np.uint8), cv2.DIST_L2, 3)
    return int((pred & (dt_ref <= tol)).sum()), int(pred.sum()), int((ref & (dt_pred <= tol)).sum()), int(ref.sum())
def _edges():
    REFS = {i: thin(hed_map(BGR[i]) >= 0.5) for i in JOINTS}
    # (1) gating leaves routed frames unchanged
    same = max(float(np.abs(hed_forward(hed, BGR[i]) - hed_forward(hed, BGR[i])).max()) for i in JOINTS[:5])
    methods = {}
    for sc in [0.75, 0.5]:
        methods[f"HED at {int(sc * 100)}% input size"] = ({i: hed_map(BGR[i], sc) for i in JOINTS}, np.linspace(0.1, 0.9, 17), "prob")
    grays = {i: cv2.cvtColor(BGR[i], cv2.COLOR_BGR2GRAY) for i in JOINTS}
    methods["Canny"] = (grays, np.arange(30, 260, 10), "canny")
    sob = {i: np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)) for i, g in grays.items()}
    sob = {i: v / (v.max() + 1e-6) for i, v in sob.items()}
    methods["Sobel magnitude"] = (sob, np.linspace(0.05, 0.8, 16), "prob")
    tol = CONFIG["EDGE_TOL_PX"]; rows = []
    for name, (maps, ths, kind) in methods.items():
        best = (0, None, 0, 0)
        for th in ths:
            mp = mt = rp = rt = 0
            for i in JOINTS:
                pred = (cv2.Canny(maps[i], 0.4 * th, th) > 0) if kind == "canny" else thin(maps[i] >= th)
                a, b_, c_, d_ = bf_counts(pred, REFS[i], tol); mp += a; mt += b_; rp += c_; rt += d_
            P, Rr = mp / max(mt, 1), rp / max(rt, 1); F_ = 2 * P * Rr / max(P + Rr, 1e-9)
            if F_ > best[0]:
                best = (F_, th, P, Rr)
        rows.append({"method": name, "ods_F": best[0], "precision": best[2], "recall": best[3], "threshold": best[1]})
    # latency of each alternative on the CPU (per frame)
    lat_m = {"HED at 75% input size": lambda: hed_map(BGR[0], 0.75), "HED at 50% input size": lambda: hed_map(BGR[0], 0.5),
             "Canny": lambda: cv2.Canny(cv2.cvtColor(BGR[0], cv2.COLOR_BGR2GRAY), 40, 100),
             "Sobel magnitude": lambda: np.hypot(cv2.Sobel(grays[JOINTS[0]], cv2.CV_32F, 1, 0), cv2.Sobel(grays[JOINTS[0]], cv2.CV_32F, 0, 1))}
    for r in rows:
        t_ = time_calls(lambda _: lat_m[r["method"]](), None, 2, CONFIG["HED_REPEATS"])
        r["latency_ms"] = 1e3 * t_.mean(); r["S_vs_HED"] = L_HED / t_.mean()
    return pd.DataFrame(rows), same
edge_df, same_diff = cached("edge_fidelity", _edges)
best_cpu = e2e_df[(e2e_df.config == cfg_str(BEST)) & (e2e_df.setting == "CPU")]
gated = {"method": f"Gated HED ({cfg_str(BEST)})", "ods_F": float(cv_mean.loc[BEST].recall_r100), "precision": 1.0,
         "recall": float(cv_mean.loc[BEST].recall_r100), "threshold": np.nan,
         "latency_ms": 1e3 * float(best_cpu.L_pipeline_s.iloc[0]) if len(best_cpu) else np.nan,
         "S_vs_HED": float(best_cpu.S_measured.iloc[0]) if len(best_cpu) else np.nan}
edge_tbl = pd.concat([pd.DataFrame([{"method": "HED, full resolution (reference)", "ods_F": 1.0, "precision": 1.0, "recall": 1.0,
                                     "latency_ms": 1e3 * L_HED, "S_vs_HED": 1.0}, gated]), edge_df], ignore_index=True)
save_table(edge_tbl.round(4), "edge_fidelity")
record("gating_output_identical_max_abs_diff", same_diff)

## 24. Speed-up versus prevalence (Reviewer 3)
The labelled frames are a sample of the recordings, so the joint rate in continuous production may differ from 9.4%. With the measured recall r and FPR f of each gate, S(pi) = 1/(rho + pi r + (1 - pi) f).

In [ ]:
prev_rows = []
pis = sorted(set(CONFIG["PREVALENCES"] + [round(PI, 4)]))
for tag, c in SEL.items():
    m_ = cv_mean.loc[c]
    for name, gdev, hdev in SETTINGS:
        rho_ = rho_setting(c[0], c[1], gdev, hdev)
        for p_ in pis:
            prev_rows.append({"config": cfg_str(c), "role": tag, "setting": name, "prevalence": p_,
                              "S": 1 / (rho_ + p_ * m_.recall_r100 + (1 - p_) * m_.fpr_r100), "bound_1_over_pi": 1 / p_})
prev_df = pd.DataFrame(prev_rows); save_table(prev_df.round(3), "prevalence_sensitivity")
fig, ax = plt.subplots(figsize=(5.6, 3.4)); pp = np.linspace(0.005, 0.3, 200)
for c, ls in zip(dict.fromkeys(SEL.values()), ["-", "-.", "--"]):
    m_ = cv_mean.loc[c]
    for name, gdev, hdev in SETTINGS:
        rho_ = rho_setting(c[0], c[1], gdev, hdev)
        ax.plot(100 * pp, 1 / (rho_ + pp * m_.recall_r100 + (1 - pp) * m_.fpr_r100), color=sty[name], ls=ls, lw=1.3,
                label=f"{c[0]}@{c[1]}, {name}")
ax.axvline(100 * PI, color="#555555", lw=0.7, ls=":")
ax.set_xlabel("Share of joint frames in the stream, $\\pi$ (%)"); ax.set_ylabel("Predicted speed-up S")
ax.legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_prevalence_sensitivity"); plt.show()

## 25. Release files (Reviewer 1.3, Reviewer 3)
`release/` contains everything needed to reproduce the evaluation without the confidential frames: fold assignments of every repetition, per-fold results of every configuration, out-of-fold scores of the selected gates, and all hyperparameters.

In [ ]:
fa = meta[["file", "recording", "station", "label"]].copy()
for seed in CONFIG["CV_SEEDS"]:
    f_ = np.full(len(y), -1)
    for k, (_, te) in enumerate(splitter("group", CONFIG["N_FOLDS"], seed).split(np.zeros(len(y)), y, groups)):
        f_[te] = k
    fa[f"fold_seed{seed}"] = f_
fa.to_csv(os.path.join(DIRS["release"], "fold_assignments.csv"), index=False)
pf_df.to_csv(os.path.join(DIRS["release"], "per_fold_results.csv"), index=False)
for tag, c in SEL.items():
    d = meta[["file", "recording", "station", "label"]].copy()
    for seed in CONFIG["CV_SEEDS"]:
        oof, _, taus = OOF[(*c, seed)]
        d[f"score_seed{seed}"] = oof; d[f"tau_seed{seed}"] = taus[1.0]
    d.to_csv(os.path.join(DIRS["release"], f"oof_scores_{tag}.csv"), index=False)
HYPER = [
    ("Input", "Frame size (height x width)", f"{H_IMG} x {W_IMG} px, RGB; half-size gates: {input_shape(0.5)[0]} x {input_shape(0.5)[1]} px"),
    ("Backbones", "Weights / pooling", "ImageNet, frozen, global average pooling"),
    ("All heads", "Training data", "training-fold frames + their horizontal mirror images; features standardized (z-score) on the training fold"),
    ("LR", "Model", f"L2-regularized logistic regression, C = {CONFIG['C']}, solver L-BFGS, max. 5000 iterations"),
    ("Weighted LR", "Class weights", "inverse class frequency (n / (2 n_c))"),
    ("LR + SMOTE", "Oversampling", f"SMOTE, k = {CONFIG['SMOTE_K']} nearest neighbours, in the standardized feature space after mirroring, until both classes are equal in size"),
    ("Weighted focal", "Loss / optimizer", f"weighted binary focal loss, gamma = {CONFIG['FOCAL_GAMMA']}, alpha = {CONFIG['FOCAL_ALPHA']}, inverse-frequency class weights, L2 = {CONFIG['FOCAL_L2']}; full-batch L-BFGS-B (max. 500 iterations), zero initialization; no learning rate or batch size"),
    ("One-class Mahalanobis", "Model", f"PCA to {CONFIG['PCA_DIM']} components + Ledoit-Wolf covariance, background frames only"),
    ("One-class kNN", "Model", f"mean cosine distance to k = {CONFIG['KNN_K']} nearest background frames"),
    ("HOG gate", "Features", "grayscale at half size, 9 orientations, 8 x 8 px cells, 2 x 2 cell blocks"),
    ("ImgStats gate", "Features", "32-bin intensity histogram, Canny (50/150) edge density and mean Sobel magnitude on an 8 x 4 grid"),
    *([("Fine-tuned gate", "Training", f"MobileNetV3-Small, all layers trainable (batch-norm statistics frozen), sigmoid output, class-weighted binary cross-entropy, Adam, learning rate {CONFIG['FT_LR']}, batch {CONFIG['FT_BATCH']}, {CONFIG['FT_EPOCHS']} epochs with random horizontal flips")] if CONFIG["FINETUNE"] else []),
    ("Evaluation", "Cross-validation", f"stratified recording-grouped {CONFIG['N_FOLDS']}-fold, {len(CONFIG['CV_SEEDS'])} repetitions (seeds {CONFIG['CV_SEEDS']})"),
    ("Threshold", "Selection", f"inner recording-grouped {CONFIG['INNER_FOLDS']}-fold CV inside each training fold, recall target 100%, margin to next lower background score"),
    ("Nested selection", "Inner CV", f"{CONFIG['NESTED_INNER_FOLDS']}-fold recording-grouped CV inside each outer training fold, {len(CANDIDATES)} candidates"),
    ("Bootstrap", "Resamples", f"{CONFIG['BOOTSTRAP']}, resampling recordings"),
    ("Latency", "Protocol", f"batch 1, {CONFIG['LAT_WARMUP']} warm-up calls, {CONFIG['LAT_REPEATS']} timed calls per gate, {CONFIG['HED_REPEATS']} per HED"),
]
hyper_df = pd.DataFrame(HYPER, columns=["component", "setting", "value"])
hyper_df.to_csv(os.path.join(DIRS["release"], "hyperparameters.csv"), index=False); save_table(hyper_df, "hyperparameters", show=False)
json.dump(CONFIG, open(os.path.join(DIRS["release"], "config.json"), "w"), indent=2)
print("release files:", sorted(os.listdir(DIRS["release"])))

## 26a. LaTeX table and macro generator (used by the export cell)

In [ ]:
"""Builds LaTeX macros and tables for the revised MLWA manuscript (v4) from the CSV/JSON outputs of
MLWA_experiments_v4.ipynb.  Usage: build(src_dir, out_dir) -> out_dir/results_v4.tex, out_dir/tables/*.tex
The same source is embedded in the notebook (Section 26a)."""
import os, json
import numpy as np
import pandas as pd

GATE = {"MobileNetV3Small": "MobileNetV3-S", "MobileNetV3Large": "MobileNetV3-L",
        "EfficientNetB0": "EfficientNet-B0", "ResNet50": "ResNet50", "HOG": "HOG", "ImgStats": "Image statistics"}
HANDCRAFTED = ("HOG", "ImgStats")
STRAT = {"LR": "LR", "LR-weighted": "Weighted LR", "LR-SMOTE": "LR + SMOTE", "Focal-weighted": "Weighted focal",
         "OneClass-Mahalanobis": "One-class Mahalanobis", "OneClass-kNN": "One-class $k$NN"}
SETTING = {"CPU": "CPU", "hybrid": "Hyb", "GPU": "GPU"}
SETTING_TXT = {"CPU": "CPU (gate + HED on CPU)", "hybrid": "Hybrid (gate on GPU, HED on CPU)", "GPU": "GPU (gate + HED on GPU)"}
NA = r"\tbd{n/a}"
MANUSCRIPT_MACROS = ['ApparentCPURecall', 'ApparentCPUS', 'BestCPHigh', 'BestCPLow', 'BestCalibAFPR', 'BestCalibARecall', 'BestCalibBRecall', 'BestCalibCFPR', 'BestCalibCRecall', 'BestDetected', 'BestFPR', 'BestFPRHalf', 'BestFPRHi', 'BestFPRLo', 'BestFrameFPR', 'BestFrameS', 'BestGroupFPR', 'BestGroupS', 'BestHead', 'BestMissHalf', 'BestMixedFPRR', 'BestMixedPRAUC', 'BestName', 'BestPRAUC', 'BestPRAUCFold', 'BestPRAUCFoldSD', 'BestQ', 'BestRecall', 'BestRecallHalf', 'BestRecallLB', 'BestRecallLegacy', 'BestRhoCPU', 'BestSCPU', 'BestSCPUHi', 'BestSCPULo', 'BestSGPU', 'BestStationFPR', 'BestStationMissed', 'BestStationRecall', 'CPUCores', 'CPUModel', 'CVVersion', 'EdgeCannyF', 'EdgeHEDHalfF', 'EdgeHEDHalfS', 'EdgeHEDTQF', 'EdgeHEDTQS', 'EdgeSobelF', 'EdgeTol', 'EffFullCPUms', 'EffFullGFLOPs', 'EtoEBestCPUFPS', 'EtoEBestCPUS', 'EtoEBestCPUSD', 'EtoEBestCPUms', 'EtoEBestGPUFPS', 'EtoEBestGPUS', 'EtoEBestGPUms', 'EtoEBestHybS', 'EtoEBestHybms', 'EtoEGBestHybS', 'EtoEGBestHybSpred', 'EtoEMaxErr', 'EtoEMeanAbsErr', 'EtoERefCPUS', 'EtoERefHybS', 'FLOPsLatCorr', 'FPSHedCPU', 'FastHead', 'FastName', 'FastRecall', 'FastSCPU', 'FocalAlpha', 'FocalGamma', 'FocalLtwo', 'GBestCPUms', 'GBestFPR', 'GBestHead', 'GBestName', 'GBestPRAUC', 'GBestQ', 'GBestRecall', 'GBestRhoGPU', 'GBestSCPU', 'GBestSGPU', 'GFLOPsRange', 'GPUModel', 'GPUmsMax', 'GPUmsMin', 'GatingIdenticalDiff', 'HED', 'HOGCPUms', 'HOGFPR', 'HOGRecall', 'HOGSCPU', 'HedGFLOPs', 'HedGPUSpeedup', 'HedOverResFLOPs', 'HedOverResLat', 'HedPortDiff', 'ImgStatsCPUms', 'ImgStatsFPR', 'ImgStatsRecall', 'ImgStatsSCPU', 'JackBestMax', 'JackBestMed', 'JackBestMissedFull', 'JackBestMissedWorst', 'JackBestN', 'JackBestShare', 'JointRate', 'LHedCPUms', 'LHedCPUsd', 'LHedGPUms', 'LRC', 'LegacyFixedTau', 'LegacyFixedValFP', 'LegacyFixedValQ', 'LegacyStarTau', 'LogoBestAugRecall', 'LogoBestCleanRecall', 'LogoRefAugRecall', 'LogoRefCleanRecall', 'MNSHalfPRAUC', 'MirFPROn', 'MirFPRPlainOn', 'MixedFrames', 'MixedJoints', 'NAugCopies', 'NBackground', 'NBoot', 'NCalibSplits', 'NCandidates', 'NConfigs', 'NEtoERuns', 'NFolds', 'NFoldsTotal', 'NFrames', 'NHeadTests', 'NInner', 'NJoints', 'NJointsOnlyJointStations', 'NNestedInner', 'NNestedSeeds', 'NPerturb', 'NRecJoints', 'NRecordings', 'NRepeats', 'NStations', 'NStratFullRecall', 'NStratSup', 'NSup', 'NSupFullRecall', 'NestedCPUFPR', 'NestedCPUMissedTotal', 'NestedCPUNDistinct', 'NestedCPUNFolds', 'NestedCPURecall', 'NestedCPURecallMin', 'NestedCPUS', 'NestedCPUTop', 'OneClassBestFPR', 'OneClassBestName', 'OneClassBestPRAUC', 'OneClassBestRecall', 'PertBestAugMeanFPR', 'PertBestAugMeanRecall', 'PertBestAugWorstName', 'PertBestAugWorstRecall', 'PertBestCleanGlareFPR', 'PertBestCleanGlareRecall', 'PertBestCleanMaxFPR', 'PertBestCleanMeanFPR', 'PertBestCleanMeanRecall', 'PertBestCleanN', 'PertBestCleanNFull', 'PertBestCleanWorstName', 'PertBestCleanWorstRecall', 'PrevBestCPUHigh', 'PrevBestCPULow', 'PrevHigh', 'PrevLow', 'RefCPUms', 'RefCalibARecall', 'RefCalibCRecall', 'RefFPR', 'RefFrameFPR', 'RefFrameS', 'RefGroupFPR', 'RefGroupS', 'RefHead', 'RefName', 'RefRecall', 'RefRhoCPU', 'RefSCPU', 'RefSGPU', 'RefStationFPR', 'RefStationMissed', 'ResHalfCPUms', 'ResHalfGFLOPs', 'RhoCPUMax', 'RhoCPUMin', 'RhoFullMax', 'RhoFullMin', 'RhoGPUMaxPct', 'RuleBestConfBound', 'RuleBestConfRecall', 'RuleBestFixedRecall', 'RuleBestNoMarginRecall', 'RuleBestRninetyfiveRecall', 'RuleBestRninetyfiveSCPU', 'SmoteK', 'StOnlyMixedFPRR', 'StOnlyMixedPRAUC', 'StationFPRMax', 'StationFPRMin', 'StationsNoJoints', 'StationsOnlyJoints', 'StratRecallMin', 'StratSMax', 'StratSMin', 'StreamBestCPUMaxCam', 'SupMinPRAUCBestGate', 'TFVersion', 'TauBestMax', 'TauBestMean', 'TauBestMin', 'TauBestSD', 'Platform']


def _cfg(g, s):
    return GATE.get(g, g) if g in HANDCRAFTED else f"{GATE.get(g, g)} ({float(s):g}$\\times$)"


def _cfg_str(c):
    """'EfficientNetB0@0.5 LR-weighted' -> ('EfficientNet-B0 (0.5x)', 'Weighted LR')"""
    gs, st = c.split(" ", 1)
    g, s = gs.split("@")
    return _cfg(g, s), STRAT.get(st, st)


def _pm(x):
    """'0.123 ± 0.004' -> (0.123, 0.004)"""
    if isinstance(x, (float, int, np.floating, np.integer)) and not isinstance(x, bool):
        return float(x), 0.0
    if not isinstance(x, str) or "±" not in x:
        return np.nan, np.nan
    a, b = x.split("±")
    return float(a), float(b)


def _num(v, nd):
    s = f"{v:.{nd}f}"
    return s[1:] if s.startswith("-") and float(s) == 0 else s          # never print "-0"


def _fmt_pm(x, scale=1.0, nd=1):
    """mean +- SD, SD always shown (Reviewer 1, comment 8)"""
    m, s = _pm(x)
    if np.isnan(m):
        return "--"
    return f"{_num(m * scale, nd)} $\\pm$ {_num(abs(s) * scale, nd)}"


def _f(x, nd=1, scale=1.0):
    try:
        if x is None or np.isnan(float(x)):
            return "--"
    except (TypeError, ValueError):
        return str(x)
    return _num(float(x) * scale, nd)


def _err(x):
    v = float(f"{x:.0f}")
    return "0" if v == 0 else f"{v:+.0f}"


def _write(path, body):
    with open(path, "w") as f:
        f.write(body)


def _tabular(cols, header, rows, note=None):
    lines = [r"\begin{adjustbox}{max width=\linewidth}", r"\begin{tabular}{@{}" + cols + r"@{}}", r"\toprule",
             " & ".join(header) + r" \\", r"\midrule"]
    lines += [" & ".join(r) + r" \\" for r in rows]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}"]
    if note:
        lines += [r"\par\vspace{2pt}{\footnotesize " + note + r"\par}"]
    return "\n".join(lines) + "\n"


def build(src, out):
    os.makedirs(os.path.join(out, "tables"), exist_ok=True)
    rd = lambda n: (pd.read_csv(os.path.join(src, n), dtype={"station": str}) if os.path.exists(os.path.join(src, n)) else None)
    T = lambda name, body: _write(os.path.join(out, "tables", name + ".tex"), body)
    cfg = json.load(open(os.path.join(src, "config.json")))
    env = json.load(open(os.path.join(src, "environment.json")))
    sel = json.load(open(os.path.join(src, "selection.json")))
    best, ref = tuple(sel["best"]), tuple(sel["reference"])
    gbest = tuple(sel["gpu_best"]) if sel.get("gpu_best") else None
    key = lambda c: f"{c[0]}@{c[1]} {c[2]}"
    best_s, ref_s, gbest_s = key(best), key(ref), (key(gbest) if gbest else None)
    acc = tuple(sel["most_accurate"]) if sel.get("most_accurate") else None
    acc_s = key(acc) if acc else None
    ROLE = {"best": best, "gpu_best": gbest, "reference": ref}
    TAG = {"best": "Best", "gpu_best": "GBest", "reference": "Ref"}
    ds, st = rd("dataset_summary.csv").iloc[0], rd("frames_per_station.csv")
    cx, lat, cvm, cvs = rd("complexity.csv"), rd("latency_cpu_gpu.csv"), rd("cv_grouped_mean.csv"), rd("cv_grouped_sd.csv")
    bpg, strat, rob = rd("best_per_gate.csv"), rd("strategies_on_best_gate.csv"), rd("robustness_protocols.csv")
    mir, leg, pst, conf = rd("mirror_stress_test.csv"), rd("legacy_gate_threshold.csv"), rd("per_station_logo.csv"), rd("confound_check.csv")
    gpu = "GPU" in set(lat.device)
    M = {}
    N_hand = len([h for h in cfg.get("HANDCRAFTED", []) if h in set(cvm.gate)])

    # ---------------- dataset, protocol, environment ----------------
    pi = ds.joints / ds.frames
    n_cnn = len(cfg["BACKBONES"]) * len(cfg["SCALES"])
    M.update({"NFrames": int(ds.frames), "NJoints": int(ds.joints), "NBackground": int(ds.background),
              "JointRate": f"{100 * pi:.1f}", "NRecordings": int(ds.recordings), "NRecJoints": int(ds.recordings_with_joints),
              "NStations": int(ds.stations), "NFolds": cfg["N_FOLDS"], "NInner": cfg["INNER_FOLDS"],
              "NRepeats": len(cfg["CV_SEEDS"]), "NGates": n_cnn, "NStrategies": len(cfg["STRATEGIES"]),
              "NConfigs": n_cnn * len(cfg["STRATEGIES"]), "NHand": N_hand,
              "NCandidates": (n_cnn + N_hand) * len(cfg["STRATEGIES"]), "NFoldsTotal": cfg["N_FOLDS"] * len(cfg["CV_SEEDS"]),
              "NBoot": cfg.get("BOOTSTRAP", 0), "NNestedInner": cfg.get("NESTED_INNER_FOLDS", 4),
              "NNestedSeeds": len(cfg.get("NESTED_SEEDS", [])), "NCalibSplits": cfg.get("CALIB_SPLITS", 0),
              "NAugCopies": cfg.get("AUG_COPIES", 0), "NPerturb": len(cfg.get("PERTURBATIONS", [])),
              "EdgeTol": cfg.get("EDGE_TOL_PX", 2), "SmoteK": cfg.get("SMOTE_K", 5), "LRC": f"{cfg['C']:g}",
              "FocalGamma": f"{cfg['FOCAL_GAMMA']:g}", "FocalAlpha": f"{cfg['FOCAL_ALPHA']:g}", "FocalLtwo": f"{cfg['FOCAL_L2']:g}",
              "FTEpochs": cfg.get("FT_EPOCHS", 0), "FTLR": f"{cfg.get('FT_LR', 0):g}", "FTBatch": cfg.get("FT_BATCH", 0),
              "CPUModel": env["cpu_model"].replace("(R)", "").replace("CPU ", "").replace("  ", " "),
              "CPUCores": env["cpu_logical_cores"], "Platform": env.get("platform") or "Google Colaboratory",
              "GPUModel": (env.get("gpu_model") or "none").split(",")[0].replace("Tesla ", ""), "TFVersion": env["tensorflow"],
              "CVVersion": env["opencv"]})
    only_joint = st[(st.joints == st.frames) & (st.frames > 0)].station.tolist()
    no_joint = st[st.joints == 0].station.tolist()
    M["StationsOnlyJoints"] = ", ".join(only_joint) if only_joint else "none"
    M["NStationsOnlyJoints"] = len(only_joint)
    M["NJointsOnlyJointStations"] = int(st[st.station.isin(only_joint)].joints.sum())
    M["StationsNoJoints"] = ", ".join(no_joint) if no_joint else "none"

    # ---------------- complexity and latency ----------------
    hed = lat[(lat.model == "HED") & (lat.device == "CPU")].iloc[0]
    hed_g = lat[lat.model.str.startswith("HED (PyTorch")]
    port = sel.get("hed_gpu_port") or {}
    hedfl = float(cx[cx.gate.str.startswith("HED")].GFLOPs.iloc[0])
    M.update({"LHedCPUms": f"{hed.mean_ms:.0f}", "LHedCPUsd": f"{hed.sd_ms:.0f}", "HedGFLOPs": f"{hedfl:.1f}",
              "FPSHedCPU": f"{1000 / hed.mean_ms:.1f}", "HedPort": "yes" if port.get("ok") else "no",
              "HedPortDiff": (f"{port['max_abs_diff']:.1e}".replace("e-0", r"\times 10^{-").replace("e-", r"\times 10^{-") + "}")
              if port.get("max_abs_diff") is not None and np.isfinite(port.get("max_abs_diff", np.inf)) else NA})
    if len(hed_g):
        M.update({"LHedGPUms": f"{hed_g.mean_ms.iloc[0]:.1f}", "FPSHedGPU": f"{1000 / hed_g.mean_ms.iloc[0]:.0f}",
                  "HedGPUSpeedup": f"{hed.mean_ms / hed_g.mean_ms.iloc[0]:.0f}"})

    def lrow(g, s, dev):
        r = lat[(lat.model == g) & (lat.scale == float(s)) & (lat.device == dev)]
        return r.iloc[0] if len(r) else None

    def crow(g, s):
        r = cx[(cx.gate == g) & (cx.scale == float(s))]
        return r.iloc[0] if len(r) else None

    for role, c in ROLE.items():
        if c is None:
            continue
        tg = TAG[role]
        M[f"{tg}Name"] = _cfg(c[0], c[1]); M[f"{tg}Head"] = STRAT[c[2]]
        cr = crow(c[0], c[1])
        M[f"{tg}GFLOPs"] = f"{cr.GFLOPs:.2f}" if cr is not None else "--"
        M[f"{tg}ParamsM"] = f"{cr.params_M:.1f}" if cr is not None else "--"
        M[f"{tg}CPUms"] = f"{lrow(c[0], c[1], 'CPU').mean_ms:.1f}"; M[f"{tg}RhoCPU"] = f"{lrow(c[0], c[1], 'CPU').rho:.3f}"
        if gpu:
            lg = lrow(c[0], c[1], "GPU")
            M[f"{tg}GPUms"] = f"{lg.mean_ms:.1f}"; M[f"{tg}RhoGPU"] = f"{lg.rho:.3f}"
            if len(hed_g):
                M[f"{tg}RhoFull"] = f"{lg.mean_ms / hed_g.mean_ms.iloc[0]:.2f}"
    gates = lat[~lat.model.str.startswith("HED") & ~lat.model.isin(HANDCRAFTED)]
    gfl = cx[~cx.gate.str.startswith("HED") & ~cx.gate.isin(HANDCRAFTED)].GFLOPs
    M["GFLOPsRange"] = f"{gfl.max() / gfl.min():.0f}"
    M["RhoCPUMin"], M["RhoCPUMax"] = f"{gates[gates.device == 'CPU'].rho.min():.3f}", f"{gates[gates.device == 'CPU'].rho.max():.3f}"
    if gpu:
        g = gates[gates.device == "GPU"]
        M["RhoGPUMin"], M["RhoGPUMax"] = f"{g.rho.min():.3f}", f"{g.rho.max():.3f}"
        M["GPUmsMin"], M["GPUmsMax"] = f"{g.mean_ms.min():.1f}", f"{g.mean_ms.max():.1f}"
        M["RhoGPUMaxPct"] = f"{100 * g.rho.max():.1f}"
        if len(hed_g):
            M["RhoFullMin"], M["RhoFullMax"] = f"{g.mean_ms.min() / hed_g.mean_ms.iloc[0]:.2f}", f"{g.mean_ms.max() / hed_g.mean_ms.iloc[0]:.2f}"
    for h in HANDCRAFTED:
        lh = lrow(h, 1.0, "CPU")
        if lh is not None:
            M[f"{h}CPUms"] = f"{lh.mean_ms:.1f}"; M[f"{h}Rho"] = f"{lh.rho:.3f}"
    try:
        rf = crow("ResNet50", 1.0)
        M["HedOverResFLOPs"] = f"{hedfl / rf.GFLOPs:.1f}"
        M["HedOverResLat"] = f"{hed.mean_ms / lrow('ResNet50', 1.0, 'CPU').mean_ms:.1f}"
    except (IndexError, AttributeError, TypeError):
        M["HedOverResFLOPs"] = M["HedOverResLat"] = NA
    cpu = gates[gates.device == "CPU"].merge(cx, left_on=["model", "scale"], right_on=["gate", "scale"])
    M["FLOPsLatCorr"] = f"{np.corrcoef(np.log(cpu.GFLOPs), np.log(cpu.mean_ms))[0, 1]:.2f}"
    for k_, g_, s_ in [("EffFull", "EfficientNetB0", 1.0), ("ResHalf", "ResNet50", 0.5)]:
        try:
            M[k_ + "GFLOPs"], M[k_ + "CPUms"] = f"{crow(g_, s_).GFLOPs:.2f}", f"{lrow(g_, s_, 'CPU').mean_ms:.1f}"
        except (IndexError, AttributeError, TypeError):
            M[k_ + "GFLOPs"] = M[k_ + "CPUms"] = NA

    rows = []
    for g_, s_ in [(g, s) for g in cfg["BACKBONES"] for s in cfg["SCALES"]]:
        c, lc, lg = crow(g_, s_), lrow(g_, s_, "CPU"), lrow(g_, s_, "GPU") if gpu else None
        r = [_cfg(g_, s_), c.input.replace("x", r"$\times$"), f"{c.params_M:.1f}", f"{c.GFLOPs:.2f}",
             f"{lc.mean_ms:.1f} $\\pm$ {lc.sd_ms:.1f}", f"{lc.rho:.3f}"]
        if gpu:
            r += [f"{lg.mean_ms:.1f} $\\pm$ {lg.sd_ms:.1f}", f"{lg.rho:.3f}"]
        rows.append(r)
    for h in HANDCRAFTED:
        lc = lrow(h, 1.0, "CPU")
        if lc is None:
            continue
        r = [GATE[h] + r"$^{b}$", "--", "--", "--", f"{lc.mean_ms:.1f} $\\pm$ {lc.sd_ms:.1f}", f"{lc.rho:.3f}"]
        if gpu:
            r += ["--", "--"]
        rows.append(r)
    hr = ["HED (stage 2)", cx[cx.gate.str.startswith("HED")].input.iloc[0].replace("x", r"$\times$"), "14.7", f"{hedfl:.1f}",
          f"{hed.mean_ms:.0f} $\\pm$ {hed.sd_ms:.0f}", "1"]
    if gpu:
        hr += ([f"{hed_g.mean_ms.iloc[0]:.1f} $\\pm$ {hed_g.sd_ms.iloc[0]:.1f}$^{{a}}$", "--"] if len(hed_g) else ["--", "--"])
    rows.append(hr)
    head = ["Gate (input scale)", "Input", "Params (M)", "GFLOPs", "CPU (ms)", r"$\rho_{\mathrm{CPU}}$"]
    if gpu:
        head += ["GPU (ms)", r"$\rho_{\mathrm{Hyb}}$"]
    note = (r"Latency per frame, batch size 1, mean $\pm$ SD. $\rho = L_{\mathrm{gate}}/L_{\mathrm{HED,CPU}}$; in the GPU column, "
            r"$\rho_{\mathrm{Hyb}}$ is the GPU gate relative to HED on the CPU (hybrid setting). 1 multiply--add = 2 FLOPs. "
            + (r"$^{a}$HED on the GPU through the PyTorch port of the Caffe model (Section~\ref{sec:setup}). " if len(hed_g) else "")
            + r"$^{b}$Hand-crafted features computed on the CPU.")
    T("tab_complexity", _tabular("l" + "r" * (len(head) - 1), head, rows, note))

    # ---------------- design space: best head per gate ----------------
    bp = bpg.copy()
    bp["S_m"] = bp.S_CPU_r100.map(lambda x: _pm(x)[0])
    rows = []
    for _, r in bp.sort_values("S_m", ascending=False).iterrows():
        rr = [_cfg(r.gate, r.scale), STRAT[r["best strategy"]], _fmt_pm(r.pr_auc, 1, 3), _fmt_pm(r.get("pr_auc_fold", np.nan), 1, 3),
              _fmt_pm(r.fpr_at_r100, 100, 1), _fmt_pm(r.recall_r100, 100, 1), _fmt_pm(r.fpr_r100, 100, 1),
              _fmt_pm(r.q_r100, 100, 1), _fmt_pm(r.S_CPU_r100, 1, 1)]
        if gpu and "S_GPU_r100" in r:
            rr.append(_fmt_pm(r.S_GPU_r100, 1, 1))
        if (r.gate, float(r.scale), r["best strategy"]) == (best[0], float(best[1]), best[2]):
            rr = [r"\textbf{" + x + "}" for x in rr]; rr[0] += r"$^{\ast}$"
        elif gbest and (r.gate, float(r.scale), r["best strategy"]) == (gbest[0], float(gbest[1]), gbest[2]):
            rr[0] += r"$^{\dagger}$"
        rows.append(rr)
    head = ["Gate", "Head", "PR-AUC (pooled)", "PR-AUC (per fold)", r"FPR@R$_{100}$ (\%)", r"Recall (\%)", r"FPR (\%)", r"$q$ (\%)", r"$S_{\mathrm{CPU}}$"]
    if gpu:
        head.append(r"$S_{\mathrm{Hyb}}$")
    note = (r"Mean $\pm$ SD over " + str(M["NRepeats"]) + r" repetitions of recording-grouped " + str(M["NFolds"]) +
            r"-fold cross-validation (per-fold PR-AUC: over all " + str(M["NFoldsTotal"]) + r" test folds). "
            r"FPR@R$_{100}$: threshold-free share of background frames scoring at least as high as the lowest joint. "
            r"Recall, FPR and $q$ at the threshold selected inside each training fold for a 100\% recall target. "
            r"$S = 1/(\rho + \bar q)$ with $\bar q$ the mean over repetitions; SD of $S$ by the delta method. "
            r"$^{\ast}$Selected for the CPU (highest $S_{\mathrm{CPU}}$ with 100\% recall in every repetition); "
            r"$^{\dagger}$selected for the GPU in the same way. Selection on these same results is optimistic; Table~\ref{tab:nested} gives the nested estimate.")
    T("tab_design_space", _tabular("ll" + "r" * (len(head) - 2), head, rows, note))

    # ---------------- imbalance remedies on the best gate ----------------
    rows = []
    for _, r in strat.iterrows():
        rr = [STRAT[r.strategy], _fmt_pm(r.pr_auc, 1, 3), _fmt_pm(r.recall_r100, 100, 1), _fmt_pm(r.fpr_r100, 100, 1),
              _fmt_pm(r.q_r100, 100, 1), _fmt_pm(r.S_CPU_r100, 1, 1),
              _fmt_pm(r.get("recall_t050", np.nan), 100, 1), _fmt_pm(r.get("recall_t035", np.nan), 100, 1)]
        if r.strategy == best[2]:
            rr = [r"\textbf{" + x + "}" for x in rr]
        rows.append(rr)
    head = ["Head", "PR-AUC", r"Recall (\%)", r"FPR (\%)", r"$q$ (\%)", r"$S_{\mathrm{CPU}}$", r"Recall, $\tau$=0.5 (\%)", r"Recall, $\tau$=0.35 (\%)"]
    fx = [100 * _pm(strat[c].iloc[i])[0] for c in ("fpr_t050", "fpr_t035") if c in strat for i in range(len(strat))
          if not np.isnan(_pm(strat[c].iloc[i])[0])]
    fx_txt = ("n/a" if not fx else f"{max(fx):.1f}\\%" if f"{min(fx):.1f}" == f"{max(fx):.1f}" else f"{min(fx):.1f}--{max(fx):.1f}\\%")
    note = (r"Gate: " + M["BestName"] + r". Mean $\pm$ SD over repetitions. Columns 3--6: threshold selected for a 100\% recall target. "
            r"Last two columns: fixed thresholds (FPR " + fx_txt + r"). One-class heads output distances, so fixed thresholds do not apply.")
    T("tab_strategies", _tabular("l" + "r" * 7, head, rows, note))
    sb = strat.set_index("strategy")
    M["BestRecallHalf"] = _f(_pm(sb.loc[best[2], "recall_t050"])[0], 1, 100)
    M["BestMissHalf"] = _f(1 - _pm(sb.loc[best[2], "recall_t050"])[0], 1, 100)
    M["BestRecallLegacy"] = _f(_pm(sb.loc[best[2], "recall_t035"])[0], 1, 100)
    M["BestFPRHalf"] = _f(_pm(sb.loc[best[2], "fpr_t050"])[0], 1, 100)
    sup = sb.loc[[s for s in sb.index if not s.startswith("OneClass")]]
    Ssup = sup.S_CPU_r100.map(lambda x: _pm(x)[0])
    M["StratSMin"], M["StratSMax"] = f"{Ssup.min():.1f}", f"{Ssup.max():.1f}"
    Rsup = sup.recall_r100.map(lambda x: _pm(x)[0])
    M["StratRecallMin"] = f"{100 * Rsup.min():.1f}"
    M["NStratFullRecall"] = int((Rsup >= 0.9999).sum()); M["NStratSup"] = len(Rsup)

    # ---------------- scalar results from the full grid ----------------
    cvm["key"] = list(zip(cvm.gate, cvm.scale.astype(float), cvm.strategy)); cg = cvm.set_index("key")
    cvs["key"] = list(zip(cvs.gate, cvs.scale.astype(float), cvs.strategy)); cs = cvs.set_index("key")
    k3 = lambda c: (c[0], float(c[1]), c[2])
    for role, c in ROLE.items():
        if c is None:
            continue
        tg, r, sd = TAG[role], cg.loc[[k3(c)]].iloc[0], cs.loc[[k3(c)]].iloc[0]
        M.update({f"{tg}PRAUC": f"{r.pr_auc:.3f}", f"{tg}PRAUCFold": f"{r.get('pr_auc_fold', np.nan):.3f}",
                  f"{tg}PRAUCFoldSD": f"{sd.get('pr_auc_fold', np.nan):.3f}", f"{tg}FPRRfull": f"{100 * r.fpr_at_r100:.1f}",
                  f"{tg}Recall": f"{100 * r.recall_r100:.1f}", f"{tg}FPR": f"{100 * r.fpr_r100:.1f}", f"{tg}FPRSD": f"{100 * sd.fpr_r100:.1f}",
                  f"{tg}Q": f"{100 * r.q_r100:.1f}", f"{tg}QSD": f"{100 * sd.q_r100:.1f}", f"{tg}SCPU": f"{r.S_CPU_r100:.1f}",
                  f"{tg}SCPUSD": f"{sd.S_CPU_r100:.1f}", f"{tg}RecallSD": f"{100 * sd.recall_r100:.1f}",
                  f"{tg}MissedTotal": str(int(round(max(0.0, 1 - r.recall_r100) * M["NJoints"] * M["NRepeats"])))})
        if gpu:
            M[f"{tg}SGPU"] = f"{r.S_GPU_r100:.1f}"
    M["GBestSame"] = "yes" if gbest and k3(gbest) == k3(best) else "no"
    if acc is not None:
        ra = cg.loc[[k3(acc)]].iloc[0]
        M.update({"AccName": _cfg(acc[0], acc[1]), "AccHead": STRAT[acc[2]], "AccPRAUC": f"{ra.pr_auc:.3f}", "AccRecall": f"{100 * ra.recall_r100:.1f}",
                  "AccFPR": f"{100 * ra.fpr_r100:.1f}", "AccQ": f"{100 * ra.q_r100:.1f}", "AccSCPU": f"{ra.S_CPU_r100:.1f}",
                  "AccSame": "yes" if k3(acc) == k3(best) else "no"})
        if gpu:
            M["AccSGPU"] = f"{ra.S_GPU_r100:.1f}"; M["AccGPUms"] = f"{lrow(acc[0], acc[1], 'GPU').mean_ms:.1f}"
            M["AccCPUms"] = f"{lrow(acc[0], acc[1], 'CPU').mean_ms:.1f}"
    strict = cvm[(cvm.recall_r100 >= 0.9999)]
    supc = cvm[~cvm.strategy.str.startswith("OneClass")]
    fast = supc.sort_values("S_CPU_r100", ascending=False).iloc[0]
    M.update({"FastName": _cfg(fast.gate, fast.scale), "FastHead": STRAT[fast.strategy], "FastSCPU": f"{fast.S_CPU_r100:.1f}",
              "FastRecall": f"{100 * fast.recall_r100:.1f}", "FastFPR": f"{100 * fast.fpr_r100:.1f}",
              "FastIsBest": "yes" if k3((fast.gate, fast.scale, fast.strategy)) == k3(best) else "no"})
    M["NStrict"] = len(strict[~strict.strategy.str.startswith("OneClass")])
    rf_ = cg.loc[[k3(ref)]].iloc[0]
    M["RefRecallHalf"] = f"{100 * rf_.recall_t050:.1f}"; M["RefFPRHalf"] = f"{100 * rf_.fpr_t050:.1f}"
    M["RefRecallLegacy"] = f"{100 * rf_.recall_t035:.1f}"; M["RefFPRLegacy"] = f"{100 * rf_.fpr_t035:.1f}"
    oc = cvm[cvm.strategy.str.startswith("OneClass")]
    boc = oc.sort_values("pr_auc", ascending=False).iloc[0]
    M.update({"OneClassBestName": f"{_cfg(boc.gate, boc.scale)}, {STRAT[boc.strategy]}", "OneClassBestPRAUC": f"{boc.pr_auc:.2f}",
              "OneClassBestRecall": f"{100 * boc.recall_r100:.1f}", "OneClassBestFPR": f"{100 * boc.fpr_r100:.1f}",
              "SupMinPRAUCBestGate": f"{sup.pr_auc.map(lambda x: _pm(x)[0]).min():.3f}"})
    supv = ~cvm.strategy.str.startswith("OneClass")
    mnh = cvm[(cvm.gate == "MobileNetV3Small") & (cvm.scale == 0.5) & supv]
    if len(mnh):
        M["MNSHalfPRAUC"] = f"{mnh.pr_auc.max():.2f}"
    sup_all = cvm[supv & ~cvm.gate.isin(HANDCRAFTED)]
    M["NSupFullRecall"] = int((sup_all.recall_r100 >= 0.9999).sum()); M["NSup"] = len(sup_all)
    for h in HANDCRAFTED:          # best head of each hand-crafted gate
        hs = cvm[(cvm.gate == h) & supv]
        if len(hs):
            hb = hs.sort_values(["recall_r100", "S_CPU_r100"], ascending=False).iloc[0]
            M.update({f"{h}PRAUC": f"{hb.pr_auc:.3f}", f"{h}Recall": f"{100 * hb.recall_r100:.1f}",
                      f"{h}FPR": f"{100 * hb.fpr_r100:.1f}", f"{h}SCPU": f"{hb.S_CPU_r100:.1f}", f"{h}Head": STRAT[hb.strategy]})

    # ---------------- end-to-end (three settings, all repetitions) ----------------
    e2e, e2s = rd("end_to_end.csv"), rd("end_to_end_per_seed.csv")
    rows = []
    for _, r in e2e.iterrows():
        g_, h_ = _cfg_str(r.config)
        rows.append([g_, h_, SETTING_TXT.get(r.setting, r.setting), f"{100 * r.q:.1f}", f"{r.rho:.3f}", f"{1000 * r.L_pipeline_s:.0f}",
                     f"{r.FPS:.1f}", f"{r.S_measured:.2f} $\\pm$ {r.S_measured_sd:.2f}", f"{r.S_predicted:.2f}", _err(r["model_error_%"]),
                     f"{r.p95_ms:.0f}"])
        for role in [t for t, c_ in [("Best", best_s), ("GBest", gbest_s), ("Acc", acc_s), ("Ref", ref_s)] if c_ == r.config]:
            tg = f"EtoE{role}{SETTING.get(r.setting, r.setting)}"
            M.update({f"{tg}S": f"{r.S_measured:.1f}", f"{tg}SD": f"{r.S_measured_sd:.1f}", f"{tg}Spred": f"{r.S_predicted:.1f}",
                      f"{tg}FPS": f"{r.FPS:.1f}", f"{tg}ms": f"{1000 * r.L_pipeline_s:.0f}", f"{tg}Err": _err(r["model_error_%"]),
                      f"{tg}Q": f"{100 * r.q:.1f}", f"{tg}Pnf": f"{r.p95_ms:.0f}"})
    M["EtoEMaxErr"] = f"{e2s['model_error_%'].abs().max():.0f}"
    M["EtoEMeanAbsErr"] = f"{e2s['model_error_%'].abs().mean():.1f}"
    M["NEtoERuns"] = len(e2s)
    head = ["Gate", "Head", "Setting", r"$q$ (\%)", r"$\rho$", "Latency (ms)", "FPS", r"$S$ measured", r"$S$ predicted", r"Error (\%)", "p95 (ms)"]
    note = (r"Complete pipelines over all " + str(M["NFrames"]) + r" frames in stream order (gate on every frame, HED on the frames routed by the "
            r"cross-validated decisions), mean over the " + str(M["NRepeats"]) + r" repetitions ($S$ measured: mean $\pm$ SD). "
            r"$S$ and $\rho$ are relative to un-gated HED on the same HED device. Error = (predicted $-$ measured)/measured, for the mean values. "
            r"p95: 95th percentile of the per-frame latency.")
    T("tab_end_to_end", _tabular("lll" + "r" * 8, head, rows, note))
    sq = rd("stream_queueing.csv")
    if sq is not None and len(sq):
        rows = []
        for _, r in sq.iterrows():
            g_, h_ = _cfg_str(r.config)
            rows.append([f"{g_}, {h_}", SETTING_TXT.get(r.setting, r.setting), str(int(r.camera_fps)), f"{100 * r.utilisation:.0f}",
                         "yes" if str(r.keeps_up) == "True" else "no", f"{r.p95_delay_ms:.0f}", f"{r.max_delay_ms:.0f}"])
        T("tab_stream", _tabular("llrrlrr", ["Configuration", "Setting", "Camera FPS", r"Utilization (\%)", "Keeps up", "p95 delay (ms)", "Max. delay (ms)"],
                                 rows, r"Stream replay of repetition 1 in recording order; delay = queueing + processing (Lindley recursion). "
                                       r"Utilization $>$100\% means the queue grows without bound at that frame rate."))
        b_cpu = sq[(sq.config == best_s) & (sq.setting == "CPU")]
        ok = b_cpu[b_cpu.keeps_up.astype(str) == "True"]
        M["StreamBestCPUMaxCam"] = str(int(ok.camera_fps.max())) if len(ok) else "none"

    # ---------------- robustness protocols ----------------
    rows = []
    pname = {"frame-level 5-fold": "frame-level CV", "group-level 5-fold": "recording-grouped CV", "leave-one-station-out": "leave-one-station-out"}
    for _, r in rob.iterrows():
        g_, h_ = _cfg_str(r.config)
        rows.append([f"{g_}, {h_}", pname.get(r.protocol, r.protocol), f"{r.pr_auc:.3f}", f"{100 * r.recall_r100:.1f}",
                     f"{100 * r.fpr_r100:.1f}", f"{100 * r.q_r100:.1f}", f"{r.S_CPU_r100:.1f}"])
        tg = "Best" if r.config == best_s else ("Ref" if r.config == ref_s else "GBest")
        k_ = {"frame-level 5-fold": "Frame", "group-level 5-fold": "Group", "leave-one-station-out": "Station"}[r.protocol]
        M.update({f"{tg}{k_}Recall": f"{100 * r.recall_r100:.1f}", f"{tg}{k_}FPR": f"{100 * r.fpr_r100:.1f}",
                  f"{tg}{k_}S": f"{r.S_CPU_r100:.1f}", f"{tg}{k_}PRAUC": f"{r.pr_auc:.3f}", f"{tg}{k_}Q": f"{100 * r.q_r100:.1f}",
                  f"{tg}{k_}Missed": str(int(round((1 - r.recall_r100) * M["NJoints"])))})
    head = ["Configuration", "Protocol", "PR-AUC", r"Recall (\%)", r"FPR (\%)", r"$q$ (\%)", r"$S_{\mathrm{CPU}}$"]
    T("tab_robustness", _tabular("ll" + "r" * 5, head, rows,
      r"Frame-level and recording-grouped CV: mean over " + str(M["NRepeats"]) + r" repetitions; leave-one-station-out: one run. "
      r"Recall, FPR and $q$ at the threshold selected for a 100\% recall target; $S_{\mathrm{CPU}} = 1/(\rho_{\mathrm{CPU}} + \bar q)$."))

    if mir is not None and len(mir):
        rows = []
        for _, r in mir.iterrows():
            g_, h_ = _cfg_str(r.config)
            fa = {"True": "yes", "False": "no", True: "yes", False: "no"}.get(r.flip_augmentation, str(r.flip_augmentation))
            rows.append([f"{g_}, {h_}", fa, f"{100 * r.recall_r100:.1f}", f"{100 * r.fpr_r100:.1f}",
                         f"{100 * r.recall_mirrored:.1f}", f"{100 * r.fpr_mirrored:.1f}"])
        T("tab_mirror", _tabular("llrrrr", ["Configuration", "Flip augmentation", r"Recall (\%)", r"FPR (\%)", r"Recall, mirrored (\%)", r"FPR, mirrored (\%)"],
                                 rows, r"Test folds mirrored horizontally; thresholds unchanged. Mean over repetitions."))
        mb_ = mir[mir.config == best_s]
        on = mb_[mb_.flip_augmentation.astype(str) == "True"]; off = mb_[mb_.flip_augmentation.astype(str) == "False"]
        if len(on) and len(off):
            M.update({"MirFPROn": f"{100 * on.fpr_mirrored.iloc[0]:.1f}", "MirFPROff": f"{100 * off.fpr_mirrored.iloc[0]:.1f}",
                      "MirRecallOn": f"{100 * on.recall_mirrored.iloc[0]:.1f}", "MirRecallOff": f"{100 * off.recall_mirrored.iloc[0]:.1f}",
                      "MirFPRPlainOff": f"{100 * off.fpr_r100.iloc[0]:.1f}", "MirFPRPlainOn": f"{100 * on.fpr_r100.iloc[0]:.1f}"})

    if leg is not None and len(leg):
        rows = []
        for th in ["fixed 0.35", "fixed 0.5", "validation-selected"]:
            v = leg[(leg.threshold == th) & (leg.split == "validation")].iloc[0]
            t = leg[(leg.threshold == th) & (leg.split == "test")].iloc[0]
            rows.append([th, f"{v.tau:.3f}", f"{100 * v.recall:.0f}", f"{int(v.FP)}", f"{100 * v.q:.1f}", f"{100 * t.recall:.0f}", f"{int(t.FP)}"])
            k_ = {"fixed 0.35": "Fixed", "fixed 0.5": "Half", "validation-selected": "Star"}[th]
            M.update({f"Legacy{k_}Tau": f"{v.tau:.3f}", f"Legacy{k_}ValFP": int(v.FP), f"Legacy{k_}ValQ": f"{100 * v.q:.1f}", f"Legacy{k_}TestFP": int(t.FP)})
        T("tab_legacy", _tabular("lrrrrrr", ["Threshold", r"$\tau$", r"Val. recall (\%)", "Val. FP", r"Val. $q$ (\%)", r"Test recall (\%)", "Test FP"], rows,
                                 r"Fine-tuned ResNet50 gate of the original design, original split (validation: 8 joints, 77 background; test: 1 joint, 10 background)."))

    if pst is not None and len(pst):
        rows = [[str(r.station), str(int(r.frames)), str(int(r.joints)), _f(r.get("recall_best"), 0, 100), _f(r.get("fpr_best"), 1, 100),
                 _f(r.get("recall_ref"), 0, 100), _f(r.get("fpr_ref"), 1, 100)] for _, r in pst.iterrows()]
        T("tab_stations", _tabular("lrrrrrr", ["Held-out station", "Frames", "Joints", r"Recall, best (\%)", r"FPR, best (\%)", r"Recall, ref. (\%)", r"FPR, ref. (\%)"],
                                   rows, r"Each station is held out in turn; ``--'' = not defined (no joints or no background frames)."))
        fb = pst.fpr_best.dropna()
        if len(fb):
            M["StationFPRMin"], M["StationFPRMax"] = f"{100 * fb.min():.0f}", f"{100 * fb.max():.0f}"

    if conf is not None and len(conf):
        rows = []
        for _, r in conf.iterrows():
            nm = "Station-only baseline" if r.model.startswith("station") else ", ".join(_cfg_str(r.model))
            rows.append([nm, r.subset, str(int(r.frames)), str(int(r.joints)), _f(r.pr_auc, 3), _f(r.fpr_at_r100, 1, 100),
                         _f(r.get("recall", np.nan), 1, 100), _f(r.get("fpr", np.nan), 1, 100)])
        T("tab_confound", _tabular("llrrrrrr", ["Model", "Frames evaluated", "Frames", "Joints", "PR-AUC", r"FPR@R$_{100}$ (\%)", r"Recall (\%)", r"FPR (\%)"], rows,
                                   r"Out-of-fold scores of recording-grouped cross-validation (repetition 1). Station-only baseline: score = joint rate of the frame's station in the training folds. "
                                   r"Recall and FPR at the selected threshold."))
        mixed_rows = conf[conf.subset == "stations with both classes"]
        if len(mixed_rows):
            M["MixedFrames"], M["MixedJoints"] = int(mixed_rows.frames.iloc[0]), int(mixed_rows.joints.iloc[0])
            bm = mixed_rows[mixed_rows.model == best_s]
            if len(bm):
                M.update({"BestMixedPRAUC": _f(bm.pr_auc.iloc[0], 3), "BestMixedFPRR": _f(bm.fpr_at_r100.iloc[0], 1, 100),
                          "BestMixedRecall": _f(bm.recall.iloc[0], 1, 100), "BestMixedFPR": _f(bm.fpr.iloc[0], 1, 100)})
            so = mixed_rows[mixed_rows.model.str.startswith("station")]
            if len(so):
                M.update({"StOnlyMixedPRAUC": _f(so.pr_auc.iloc[0], 3), "StOnlyMixedFPRR": _f(so.fpr_at_r100.iloc[0], 1, 100)})

    # ---------------- uncertainty: recall bounds, bootstrap CIs ----------------
    rb, bt = rd("recall_bounds.csv"), rd("bootstrap_ci.csv")
    if rb is not None and len(rb):
        for role, tg in TAG.items():
            d = rb[rb.role == role]
            if len(d):
                w = d.sort_values("detected").iloc[0]          # worst repetition
                M.update({f"{tg}Detected": int(w.detected), f"{tg}CPLow": f"{100 * w.cp_low:.1f}", f"{tg}CPHigh": f"{100 * w.cp_high:.1f}",
                          f"{tg}RecallLB": f"{100 * w.one_sided_95_lower:.1f}"})
    if bt is not None and len(bt):
        rows, seen = [], set()
        for role, tg in TAG.items():
            d = bt[bt.role == role].set_index("metric")
            if not len(d):
                continue
            first = d.config.iloc[0] not in seen; seen.add(d.config.iloc[0])
            g_, h_ = _cfg_str(d.config.iloc[0])
            ci = lambda m, sc, nd: (f"{_f(d.loc[m, 'point'], nd, sc)} [{_f(d.loc[m, 'ci_low'], nd, sc)}, {_f(d.loc[m, 'ci_high'], nd, sc)}]"
                                    if m in d.index else "--")
            rr = [f"{g_}, {h_}", ci("pr_auc", 1, 3), ci("recall", 100, 1), ci("fpr", 100, 1), ci("q", 100, 1), ci("S_CPU", 1, 1)]
            if gpu:
                rr.append(ci("S_GPU", 1, 1))
            w = rb[rb.role == role].sort_values("detected").iloc[0] if rb is not None and len(rb[rb.role == role]) else None
            rr.append(f"{int(w.detected)}/{int(w.joints)}: [{100 * w.cp_low:.1f}, {100 * w.cp_high:.1f}]" if w is not None else "--")
            if first:
                rows.append(rr)
            for m, sc, nd in [("pr_auc", 1, 3), ("recall", 100, 1), ("fpr", 100, 1), ("q", 100, 1), ("S_CPU", 1, 1), ("S_GPU", 1, 1)]:
                if m in d.index:
                    nm = {"pr_auc": "PRAUC", "recall": "Recall", "fpr": "FPR", "q": "Q", "S_CPU": "SCPU", "S_GPU": "SGPU"}[m]
                    M[f"{tg}{nm}Lo"], M[f"{tg}{nm}Hi"] = _f(d.loc[m, "ci_low"], nd, sc), _f(d.loc[m, "ci_high"], nd, sc)
        head = ["Configuration", "PR-AUC", r"Recall (\%)", r"FPR (\%)", r"$q$ (\%)", r"$S_{\mathrm{CPU}}$"] + ([r"$S_{\mathrm{Hyb}}$"] if gpu else []) + \
               [r"Joints found: 95\% CI (\%)"]
        T("tab_uncertainty", _tabular("l" + "r" * (len(head) - 1), head, rows,
          r"Point estimate [95\% CI] from " + str(M["NBoot"]) + r" bootstrap resamples of recordings (frames of a recording kept together), "
          r"averaged over repetitions. Last column: joints detected in the worst repetition and the exact Clopper--Pearson 95\% interval for recall; "
          r"the repetitions re-use the same " + str(M["NJoints"]) + r" joints, so the interval uses $n = " + str(M["NJoints"]) + r"$."))

    # ---------------- significance tests ----------------
    tt = rd("significance_tests.csv")
    if tt is not None and len(tt):
        rows = []
        mname = {"pr_auc": "PR-AUC", "fpr_at_r100": r"FPR@R$_{100}$", "q": r"$q$"}
        for _, r in tt.iterrows():
            a_, b_ = ", ".join(_cfg_str(r.A)), ", ".join(_cfg_str(r.B))
            if r.family.startswith("heads"):
                a_, b_ = _cfg_str(r.A)[1], _cfg_str(r.B)[1]
            sc = 1 if r.metric == "pr_auc" else 100
            rows.append([r.family.replace("heads on best backbone", "heads on " + M["BestName"]), a_, b_, mname.get(r.metric, r.metric),
                         _f(r.mean_diff, 3 if sc == 1 else 1, sc), _f(r.t, 2), _f(r.p, 3), _f(r.p_holm, 3)])
        T("tab_tests", _tabular("lllrrrrr", ["Family", "A", "B", "Metric", r"Mean diff. (A$-$B)", r"$t$", r"$p$", r"$p$ (Holm)"], rows,
          r"Corrected resampled $t$-test \citep{nadeau2003inference} on the per-fold differences over " + str(M["NFoldsTotal"]) +
          r" test folds (" + str(M["NRepeats"]) + r" repetitions $\times$ " + str(M["NFolds"]) + r" folds); Holm correction within each family. "
          r"FPR@R$_{100}$ and $q$ differences in percentage points."))
        h_ = tt[tt.family.str.startswith("heads")]
        M["NHeadTests"] = len(h_); M["NHeadTestsSig"] = int((h_.p_holm < 0.05).sum())
        M["HeadTestMinP"] = _f(h_.p_holm.min(), 2) if len(h_) else NA
        s_ = tt[tt.family.str.startswith("selected")]
        M["NSelTests"] = len(s_); M["NSelTestsSig"] = int((s_.p_holm < 0.05).sum())
        for _, r in s_.iterrows():
            pair = {frozenset([best_s, ref_s]): "BestRef", frozenset([gbest_s, ref_s]): "GBestRef", frozenset([best_s, gbest_s]): "BestGBest"}.get(frozenset([r.A, r.B]))
            if pair:
                M[f"P{pair}{ {'pr_auc': 'PRAUC', 'fpr_at_r100': 'FPRR', 'q': 'Q'}[r.metric]}"] = _f(r.p_holm, 3)
    if sel.get("friedman"):
        M["FriedmanChi"], M["FriedmanP"] = _f(sel["friedman"].get("chi2"), 2), _f(sel["friedman"].get("p"), 3)
        M["FriedmanIdentical"] = "yes" if sel["friedman"].get("identical") else "no"

    # ---------------- nested selection ----------------
    nv, nf, ns = rd("nested_vs_apparent.csv"), rd("nested_selection_frequency.csv"), rd("nested_selection_per_seed.csv")
    if nv is not None and len(nv):
        rows = []
        for _, r in nv.iterrows():
            rows.append([{"CPU": "CPU rule", "GPU": "GPU rule"}.get(r.rule, r.rule), r.estimate, _f(r.recall, 1, 100), _f(r.fpr, 1, 100),
                         _f(r.q, 1, 100), _f(r.S, 2)])
            tg = ("Nested" if r.estimate.startswith("nested") else "Apparent") + {"CPU": "CPU", "GPU": "GPU"}.get(r.rule, r.rule)
            M.update({f"{tg}Recall": _f(r.recall, 1, 100), f"{tg}FPR": _f(r.fpr, 1, 100), f"{tg}Q": _f(r.q, 1, 100), f"{tg}S": _f(r.S, 1)})
        if ns is not None:
            for rule, g in ns.groupby("rule"):
                tg = "Nested" + rule
                M.update({f"{tg}SSD": _f(g.S.std(), 1), f"{tg}RecallMin": _f(g.recall.min(), 1, 100),
                          f"{tg}MissedTotal": int(g.missed.sum()), f"{tg}MissedMax": int(g.missed.max())})
        if nf is not None and len(nf):
            for rule, g in nf.groupby("rule"):
                top = g.sort_values("times_selected", ascending=False).iloc[0]
                M[f"Nested{rule}Top"] = f"{_cfg(top.gate, top.scale)} with {STRAT[top.strategy]}"
                M[f"Nested{rule}TopN"] = int(top.times_selected); M[f"Nested{rule}NDistinct"] = len(g)
                M[f"Nested{rule}NFolds"] = int(g.times_selected.sum())
        T("tab_nested", _tabular("llrrrr", ["Selection rule", "Estimate", r"Recall (\%)", r"FPR (\%)", r"$q$ (\%)", r"$S$"], rows,
          r"Apparent: the configuration selected on the cross-validation results of Table~\ref{tab:design}, evaluated on the same results. "
          r"Nested: the complete procedure (candidate assessment by inner recording-grouped CV, selection, threshold selection) is run inside each outer "
          r"training fold and evaluated on the outer test fold; mean over " + str(M["NNestedSeeds"]) + r" repetitions \citep{cawley2010overfitting}. "
          r"CPU rule: $S$ on the CPU; GPU rule: $S$ in the hybrid setting."))
        if nf is not None and len(nf):
            rows = [[{"CPU": "CPU rule", "GPU": "GPU rule"}.get(r.rule, r.rule), _cfg(r.gate, r.scale), STRAT[r.strategy], str(int(r.times_selected))]
                    for _, r in nf.sort_values(["rule", "times_selected"], ascending=[True, False]).iterrows()]
            T("tab_nested_freq", _tabular("lllr", ["Rule", "Gate", "Head", "Times selected"], rows,
              r"Configurations selected in the " + str(int(nf.groupby("rule").times_selected.sum().iloc[0])) + r" outer folds of the nested evaluation."))

    # ---------------- thresholds ----------------
    td, tj, tr_ = rd("threshold_distribution.csv"), rd("threshold_jackknife.csv"), rd("threshold_rules.csv")
    if td is not None and len(td):
        for _, r in td.iterrows():
            tg = "Best" if r.config == best_s else ("GBest" if r.config == gbest_s else "Ref")
            M.update({f"Tau{tg}Mean": _f(r["mean"], 3), f"Tau{tg}SD": _f(r["std"], 3), f"Tau{tg}Min": _f(r["min"], 3), f"Tau{tg}Max": _f(r["max"], 3)})
    if tj is not None and len(tj):
        for _, r in tj.iterrows():
            tg = "Best" if r.role == "best" else "GBest"
            M.update({f"Jack{tg}N": int(r.n_jackknife), f"Jack{tg}Med": _f(r.median_abs_tau_change, 3), f"Jack{tg}Max": _f(r.max_abs_tau_change, 3),
                      f"Jack{tg}MissedFull": int(r.missed_with_full_tau), f"Jack{tg}MissedWorst": int(r.missed_worst_jackknife_tau),
                      f"Jack{tg}Share": _f(r.share_jack_tau_missing_any, 1, 100)})
    if tr_ is not None and len(tr_):
        rows = []
        rkey = {"inner-CV + margin (ours)": "Ours", "inner-CV, no margin": "NoMargin", "split-conformal": "Conf",
                "inner-CV, r*=95%": "Rninetyfive", "fixed 0.5": "Fixed"}
        for _, r in tr_.iterrows():
            g_, h_ = _cfg_str(r.config)
            rr = [f"{g_}, {h_}", r.rule.replace("r*=95%", r"$r^\ast$=95\%"), f"{100 * r.recall_mean:.1f}", f"{100 * r.recall_min:.1f}",
                  f"{r.missed_max:.0f}", f"{100 * r.fpr_mean:.1f}", f"{100 * r.q_mean:.1f}", f"{r.S_CPU:.1f}"]
            if gpu and "S_GPU" in r:
                rr.append(f"{r.S_GPU:.1f}")
            rows.append(rr)
            tg = ("Best" if r.role == "best" else "GBest" if r.role == "gpu_best" else "Ref") + rkey.get(r.rule, "X")
            M.update({f"Rule{tg}Recall": _f(r.recall_mean, 1, 100), f"Rule{tg}RecallMin": _f(r.recall_min, 1, 100),
                      f"Rule{tg}MissedMax": f"{r.missed_max:.0f}", f"Rule{tg}FPR": _f(r.fpr_mean, 1, 100), f"Rule{tg}SCPU": _f(r.S_CPU, 1)})
            if gpu and "S_GPU" in r:
                M[f"Rule{tg}SGPU"] = _f(r.S_GPU, 1)
            if r.rule == "split-conformal" and not np.isnan(r.conformal_bound_mean):
                M[f"Rule{tg}Bound"] = _f(r.conformal_bound_mean, 1, 100)
        head = ["Configuration", "Threshold rule", r"Recall (\%)", r"Min. recall (\%)", "Max. missed", r"FPR (\%)", r"$q$ (\%)", r"$S_{\mathrm{CPU}}$"] + \
               ([r"$S_{\mathrm{Hyb}}$"] if gpu else [])
        T("tab_threshold_rules", _tabular("ll" + "r" * (len(head) - 2), head, rows,
          r"Same folds and heads; mean over repetitions, minimum recall and maximum number of missed joints over repetitions. "
          r"Split-conformal: head fitted on two thirds of the training recordings, $\tau$ = lowest joint score on the remaining third."))

    # ---------------- station calibration ----------------
    sc_ = rd("station_calibration.csv")
    if sc_ is not None and len(sc_):
        rows = []
        vkey = {"A: source threshold": "A", "B: threshold calibrated on station": "B", "C: head + threshold updated": "C"}
        for (cf, role), g in sc_.groupby(["config", "role"], sort=False):
            g_, h_ = _cfg_str(cf)
            for stn in [s for s in g.station.unique() if s != "all"] + ["all"]:
                d = g[g.station == stn].set_index("variant")
                if not len(d):
                    continue
                rr = [f"{g_}, {h_}" if stn == [s for s in g.station.unique() if s != "all"][0] else "", "all held-out stations" if stn == "all" else stn,
                      str(int(d.test_joints.iloc[0]))]
                for v in vkey:
                    rr += [_f(d.loc[v, "recall"], 0, 100) if v in d.index else "--", _f(d.loc[v, "fpr"], 1, 100) if v in d.index else "--"]
                rows.append(rr)
                if stn == "all":
                    for v, k_ in vkey.items():
                        if v in d.index:
                            tg = ("Best" if role == "best" else "GBest" if role == "gpu_best" else "Ref") + "Calib" + k_
                            M[f"{tg}Recall"], M[f"{tg}FPR"] = _f(d.loc[v, "recall"], 1, 100), _f(d.loc[v, "fpr"], 1, 100)
        T("tab_calibration", _tabular("llr" + "r" * 6, ["Configuration", "Held-out station", "Test joints", r"Recall A (\%)", r"FPR A (\%)",
                                                        r"Recall B (\%)", r"FPR B (\%)", r"Recall C (\%)", r"FPR C (\%)"], rows,
          r"Gate trained on the other stations; the held-out station's recordings are split " + str(M["NCalibSplits"]) +
          r" times at random into calibration and test recordings, and results are pooled over the test recordings of all splits. "
          r"A: source threshold; B: threshold lowered to the lowest calibration-joint score; C: head and threshold re-fitted with the calibration frames. "
          r"FPR only where the station has background frames."))

    # ---------------- perturbations ----------------
    pr, la = rd("perturbation_robustness.csv"), rd("station_transfer_augmentation.csv")
    if pr is not None and len(pr):
        PN = {"none": "none (clean)", "mirror": "mirror", "brightness-30": r"brightness $-$30\%", "brightness+30": r"brightness +30\%",
              "gamma0.6": r"gamma 0.6", "gamma1.6": r"gamma 1.6", "contrast0.6": r"contrast $\times$0.6", "noise10": r"noise $\sigma$=10",
              "noise20": r"noise $\sigma$=20", "glare": "simulated glare", "motion_blur": "motion blur (9 px)", "defocus": r"defocus ($\sigma$=2)",
              "rotate5": r"rotation 5$^\circ$", "scale_shift": r"scale 1.1 + shift 8 px", "jpeg30": "JPEG quality 30"}
        for cf in pr.config.unique():
            d = pr[pr.config == cf]
            rows = []
            for p_ in d.perturbation.unique():
                c_ = d[(d.perturbation == p_) & (d.training == "clean")]; a_ = d[(d.perturbation == p_) & (d.training == "augmented")]
                rows.append([PN.get(p_, p_), _f(c_.recall.iloc[0], 1, 100) if len(c_) else "--", _f(c_.fpr.iloc[0], 1, 100) if len(c_) else "--",
                             _f(a_.recall.iloc[0], 1, 100) if len(a_) else "--", _f(a_.fpr.iloc[0], 1, 100) if len(a_) else "--"])
            tg = "Best" if cf == best_s else ("GBest" if cf == gbest_s else "Ref")
            g_, h_ = _cfg_str(cf)
            T(f"tab_perturb_{tg.lower()}", _tabular("lrrrr", ["Test perturbation", r"Recall, clean training (\%)", r"FPR, clean training (\%)",
                                                              r"Recall, augmented training (\%)", r"FPR, augmented training (\%)"], rows,
              f"Gate: {g_}, {h_}. " + r"Thresholds selected on clean frames for a 100\% recall target; mean over " + str(M["NRepeats"]) +
              r" repetitions of recording-grouped CV. Augmented training adds " + str(M["NAugCopies"]) +
              r" random photometric/geometric copies of each training frame."))
            dp = d[~d.perturbation.isin(["none"])]
            for train, k_ in [("clean", "Clean"), ("augmented", "Aug")]:
                e = dp[dp.training == train]
                if len(e):
                    w = e.sort_values("recall").iloc[0]
                    M.update({f"Pert{tg}{k_}WorstRecall": _f(w.recall, 1, 100), f"Pert{tg}{k_}WorstName": PN.get(w.perturbation, w.perturbation),
                              f"Pert{tg}{k_}MeanRecall": _f(e.recall.mean(), 1, 100), f"Pert{tg}{k_}MeanFPR": _f(e.fpr.mean(), 1, 100),
                              f"Pert{tg}{k_}MaxFPR": _f(e.fpr.max(), 1, 100), f"Pert{tg}{k_}NFull": int((e.recall >= 0.9999).sum()),
                              f"Pert{tg}{k_}N": len(e)})
                    gl = e[e.perturbation == "glare"]
                    if len(gl):
                        M[f"Pert{tg}{k_}GlareRecall"], M[f"Pert{tg}{k_}GlareFPR"] = _f(gl.recall.iloc[0], 1, 100), _f(gl.fpr.iloc[0], 1, 100)
                cl = d[(d.perturbation == "none") & (d.training == train)]
                if len(cl):
                    M[f"Pert{tg}{k_}CleanRecall"], M[f"Pert{tg}{k_}CleanFPR"] = _f(cl.recall.iloc[0], 1, 100), _f(cl.fpr.iloc[0], 1, 100)
    if la is not None and len(la):
        rows = []
        for _, r in la.iterrows():
            g_, h_ = _cfg_str(r.config)
            rows.append([f"{g_}, {h_}", r.training, _f(r.recall, 1, 100), str(int(r.missed)), _f(r.fpr, 1, 100), _f(r.q, 1, 100)])
            tg = "Best" if r.config == best_s else ("GBest" if r.config == gbest_s else "Ref")
            M.update({f"Logo{tg}{'Aug' if r.training == 'augmented' else 'Clean'}Recall": _f(r.recall, 1, 100),
                      f"Logo{tg}{'Aug' if r.training == 'augmented' else 'Clean'}FPR": _f(r.fpr, 1, 100),
                      f"Logo{tg}{'Aug' if r.training == 'augmented' else 'Clean'}Missed": int(r.missed)})
        T("tab_logo_aug", _tabular("llrrrr", ["Configuration", "Training", r"Recall (\%)", "Missed", r"FPR (\%)", r"$q$ (\%)"], rows,
                                   r"Leave-one-station-out, threshold selected inside the training stations."))

    # ---------------- baselines ----------------
    bl, ft = rd("baselines.csv"), rd("finetuned_gate.csv")
    if bl is not None and len(bl):
        rows = []
        if not cfg.get("FINETUNE"):
            bl = bl[~bl.gate.astype(str).str.contains("fine-tuned")]
        bl = bl.drop_duplicates(subset=["gate", "head"])
        for _, r in bl.iterrows():
            gname = r.gate
            if "@" in str(gname):
                g0, rest = gname.split("@"); s0 = rest.split(" ")[0]; gname = _cfg(g0, s0) + " (frozen)"
            elif gname in GATE:
                gname = GATE[gname]
            elif gname.startswith("MobileNetV3Small fine-tuned"):
                gname = "MobileNetV3-S (1$\\times$), fine-tuned"
            rr = [gname, STRAT.get(r["head"], r["head"]), _f(r.get("pr_auc", np.nan), 3), _f(r.recall, 1, 100), _f(r.fpr, 1, 100),
                  _f(r.q, 1, 100), _f(r.S_CPU, 1)]
            if gpu and "S_GPU" in r:
                rr.append(_f(r.S_GPU, 1))
            rows.append(rr)
        head = ["Gate", "Head", "PR-AUC", r"Recall (\%)", r"FPR (\%)", r"$q$ (\%)", r"$S_{\mathrm{CPU}}$"] + ([r"$S_{\mathrm{Hyb}}$"] if gpu else [])
        T("tab_baselines", _tabular("ll" + "r" * (len(head) - 2), head, rows,
          r"All at the threshold selected for a 100\% recall target, recording-grouped CV. Random or periodic sampling finds on average the same share of joints as of frames, so full recall requires $q = 1$. "
          r"Hand-crafted gates: best head by recall, then $S_{\mathrm{CPU}}$. Fine-tuned gate: " + str(len(cfg.get("FT_SEEDS", [0]))) + r" repetition(s)."))
    if ft is not None and len(ft) and cfg.get("FINETUNE"):
        q_ = ft.q.mean(); rh = lrow("MobileNetV3Small", 1.0, "CPU")
        M.update({"FTRecall": _f(ft.recall.mean(), 1, 100), "FTMissed": int(ft.missed.sum()), "FTFPR": _f(ft.fpr.mean(), 1, 100),
                  "FTPRAUC": _f(ft.pr_auc.mean(), 3), "FTQ": _f(q_, 1, 100), "FTSCPU": _f(1 / (rh.rho + q_), 1) if rh is not None else NA})

    # ---------------- edge fidelity ----------------
    ef = rd("edge_fidelity.csv")
    if ef is not None and len(ef):
        rows = []
        for _, r in ef.iterrows():
            nm = r.method
            if nm.startswith("Gated HED"):
                nm = "Gated HED (" + M["BestName"] + ", " + M["BestHead"] + ")"
            rows.append([nm.replace("%", r"\%"), _f(r.ods_F, 3), _f(r.precision, 3), _f(r.recall, 3), _f(r.latency_ms, 1), _f(r.S_vs_HED, 1)])
            k_ = {"HED at 75% input size": "HEDTQ", "HED at 50% input size": "HEDHalf", "Canny": "Canny", "Sobel magnitude": "Sobel"}.get(r.method)
            if k_:
                M.update({f"Edge{k_}F": _f(r.ods_F, 2), f"Edge{k_}S": _f(r.S_vs_HED, 1), f"Edge{k_}ms": _f(r.latency_ms, 1)})
            if r.method.startswith("Gated"):
                M["EdgeGatedF"] = _f(r.ods_F, 2)
        T("tab_edge", _tabular("lrrrrr", ["Method", r"F (ODS)", "Precision", "Recall", "Latency (ms)", r"Speed-up vs.\ HED"], rows,
          r"Agreement with full-resolution HED (reference: HED $\geq$ 0.5, thinned) on the " + str(M["NJoints"]) + r" joint frames; boundary matching "
          r"tolerance " + str(M["EdgeTol"]) + r" px; best global threshold per method. Gated HED: output identical to HED on routed frames, "
          r"so its F equals the share of joints routed; its latency and speed-up are per frame of the stream (CPU setting). "
          r"This is agreement with HED, not accuracy against human annotation."))
    if sel.get("gating_output_identical_max_abs_diff") is not None:
        M["GatingIdenticalDiff"] = _f(sel["gating_output_identical_max_abs_diff"], 1)

    # ---------------- prevalence ----------------
    pv = rd("prevalence_sensitivity.csv")
    if pv is not None and len(pv):
        rows = []
        pis = sorted(pv.prevalence.unique())
        for (cf, setting), g in pv.groupby(["config", "setting"], sort=False):
            g_, h_ = _cfg_str(cf)
            rows.append([f"{g_}, {h_}", SETTING_TXT.get(setting, setting)] + [_f(g[g.prevalence == p_].S.iloc[0], 1) for p_ in pis])
            tg = ("Best" if cf == best_s else "GBest" if cf == gbest_s else "Ref") + SETTING.get(setting, setting)
            M[f"Prev{tg}Low"], M[f"Prev{tg}High"] = _f(g[g.prevalence == pis[0]].S.iloc[0], 1), _f(g[g.prevalence == pis[-1]].S.iloc[0], 1)
        M["PrevLow"], M["PrevHigh"] = _f(pis[0], 0, 100), _f(pis[-1], 0, 100)
        T("tab_prevalence", _tabular("ll" + "r" * len(pis), ["Configuration", "Setting"] + [r"$\pi$ = " + _f(p_, 1, 100) + r"\%" for p_ in pis], rows,
          r"Predicted speed-up $S(\pi) = 1/(\rho + \pi r + (1-\pi) f)$ with the measured recall $r$ and FPR $f$ at the selected threshold."))

    # ---------------- hyperparameters ----------------
    hp = rd("hyperparameters.csv")
    if hp is not None and len(hp):
        esc = lambda s: str(s).replace("_", r"\_").replace("%", r"\%").replace(" x ", r" $\times$ ").replace("+-", r"$\pm$")
        if not cfg.get("FINETUNE"):
            hp = hp[~hp.component.astype(str).str.startswith("Fine-tuned")]
        rows = [[esc(r.component), esc(r.setting), esc(r.value)] for _, r in hp.iterrows()]
        lines = [r"\begin{tabularx}{\linewidth}{@{}llX@{}}", r"\toprule", r"Component & Setting & Value \\", r"\midrule"]
        lines += [" & ".join(r) + r" \\" for r in rows] + [r"\bottomrule", r"\end{tabularx}"]
        T("tab_hyper", "\n".join(lines) + "\n")

    for k in MANUSCRIPT_MACROS:           # every macro used by the manuscript must exist
        M.setdefault(k, NA)
    bad = [k for k in M if not k.isalpha()]
    assert not bad, f"macro names must contain letters only: {bad}"
    lines = [f"%% Generated from MLWA_experiments_v4 outputs | {env.get('timestamp_utc', '')} UTC",
             r"\providecommand{\tbd}[1]{\colorbox{yellow}{#1}}"]
    lines += [f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in M.items()]
    _write(os.path.join(out, "results_v4.tex"), "\n".join(lines) + "\n")
    return M

build_paper_assets = build

## 26. Export
`overleaf/results_v4.tex` (every number in the manuscript as a LaTeX macro), `overleaf/tables/*.tex`, figures, `results_summary_v4.json`, `overleaf_v4.zip`, and `release/`.

In [ ]:
os.makedirs(os.path.join(DIRS["overleaf"], "figures"), exist_ok=True)
for f in os.listdir(DIRS["figures"]):
    if f.endswith(".png"):
        shutil.copy(os.path.join(DIRS["figures"], f), os.path.join(DIRS["overleaf"], "figures", f))
for f in os.listdir(DIRS["tables"]):
    if f.endswith(".csv"):
        shutil.copy(os.path.join(DIRS["tables"], f), os.path.join(DIRS["overleaf"], f))
for f in ["config.json", "environment.json"]:
    shutil.copy(os.path.join(DIRS["logs"], f), os.path.join(DIRS["overleaf"], f))
extra = {"best": list(BEST), "reference": list(REF), "gpu_best": list(GBEST) if GBEST else None,
         "most_accurate": list(ACC) if ACC else None,
         "hed_gpu_port": HED_PORT, "L_HED_gpu_s": L_HED_GPU, "friedman": RESULTS.get("friedman_heads_best_backbone"),
         "gating_output_identical_max_abs_diff": RESULTS.get("gating_output_identical_max_abs_diff")}
json.dump(extra, open(os.path.join(DIRS["overleaf"], "selection.json"), "w"), indent=2, default=float)
build_paper_assets(DIRS["overleaf"], DIRS["overleaf"])        # -> overleaf/results_v4.tex and overleaf/tables/*.tex
shutil.make_archive(os.path.join(RD, "overleaf_v4"), "zip", DIRS["overleaf"])
shutil.make_archive(os.path.join(RD, "release_v4"), "zip", DIRS["release"])
json.dump({"environment": ENV, "config": CONFIG, "results": RESULTS}, open(os.path.join(RD, "results_summary_v4.json"), "w"),
          indent=2, default=lambda o: o.item() if hasattr(o, "item") else (list(o) if isinstance(o, tuple) else str(o)))
print(open(os.path.join(DIRS["overleaf"], "results_v4.tex")).read()[:3000])
print("\nSaved:", os.path.join(RD, "overleaf_v4.zip"), os.path.join(RD, "release_v4.zip"), "and results_summary_v4.json")

### Next step
Send `results_summary_v4.json` and `overleaf_v4.zip` (in `RESULTS_DIR` on Drive). `release_v4.zip` is for the public code/data-availability repository.